# 3D Mesh Quality Control — финальное решение

**Задача.** Для каждого 3D-меша предсказать 10 бинарных меток дефектов и метку `quality`.
**Метрика.** `10 · F1(quality) + 10 · F1_weighted(10 дефектов)`, максимум 20.

## Результат

| | значение |
|---|---|
| вложенная (честная) оценка | **14.855** |
| лидерборд | **15.541** |
| состав | ансамбль трёх CNN, веса подобраны по OOF |

## Как запускать

Ноутбук рассчитан на `Runtime -> Run all` в Google Colab и не требует ручных правок.

Ключ — переменная **`USE_CACHE`** в §1:

* `True` — домашний прогон: всё уже посчитанное подтягивается с Google Drive,
  обучение пропускается, `submission.csv` собирается за минуты.
* `False` — **проверочный прогон**: ноутбук обучает все три модели с нуля,
  не используя ни одного готового артефакта. Ориентировочно 12–14 часов на A100.

Результат в обоих режимах совпадает с точностью до погрешности fp16: `random_seed`
зафиксирован, разбиение на фолды детерминировано, веса ансамбля и пороги выводятся
из OOF-предсказаний **без участия лидерборда**.

## Структура

| раздел | содержание |
|---|---|
| §0–§2 | зависимости, конфиг, кэш с возобновлением, загрузка данных с Kaggle |
| §3 | метрика соревнования |
| §4–§5 | рендеры, геометрия из `.npz`, силуэтные признаки, CLIP zero-shot |
| §6 | стратифицированные фолды |
| §7–§8 | датасет с симметриями съёмочного стенда, модель с ML-Decoder |
| §9 | обучение с прогресс-барами и метриками по эпохам |
| §10 | ансамбль: подбор весов, корреляция ошибок |
| §11 | решающее правило, пороги, согласование меток |
| §12 | `submission.csv` и проверки формата |
| §13 | визуализация: три метода, fail- и success-кейсы |
| §14 | чек-лист воспроизводимости |


## 0. Установка зависимостей

In [ ]:
!pip -q install kagglehub 'timm>=1.0.11' iterative-stratification pyarrow open_clip_torch --upgrade
!pip -q install scikit-learn pandas matplotlib tqdm pillow scipy


## 1. Конфиг и seed

Все ручки собраны здесь. `RUN_ALL_FOLDS=False` — быстрый режим (1 фолд, ~1.5 ч на T4).
Для финального сабмита ставим `RUN_ALL_FOLDS=True` (5 фолдов, ~7 ч на T4 / ~2 ч на A100).

In [ ]:
import os, gc, json, math, random, re, time, copy, warnings, shutil, itertools
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib as mpl
from sklearn.metrics import f1_score

warnings.filterwarnings('ignore')
mpl.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#fbfbfd', 'axes.grid': True,
    'grid.alpha': .25, 'grid.linestyle': '--', 'axes.spines.top': False,
    'axes.spines.right': False, 'font.size': 10, 'axes.titlesize': 11,
    'axes.titleweight': 'bold'})
PAL = {'good': '#2e9e5b', 'bad': '#d1495b', 'warn': '#e8a33d', 'main': '#2d6cdf',
       'alt': '#8a4fbd', 'grey': '#9aa0a6', 'dark': '#1f2933'}


class Reporter:
    W = 86
    _t = {}
    @classmethod
    def section(cls, t, s=''):
        print('\n' + '=' * cls.W); print(f'  {t}')
        if s: print(f'  {s}')
        print('=' * cls.W); cls._t[t] = time.time()
    @classmethod
    def done(cls, t, extra=''):
        print(f'  [OK] {t} — {(time.time() - cls._t.get(t, time.time())) / 60:.1f} мин'
              + (f' | {extra}' if extra else ''))
    @staticmethod
    def kv(**kw):
        for k, v in kw.items(): print(f'    {k:<32s} {v}')
    @staticmethod
    def ok(m): print(f'  [+] {m}')
    @staticmethod
    def warn(m): print(f'  [!] {m}')
    @staticmethod
    def fail(m): print(f'  [x] {m}')
    @staticmethod
    def bar(v, lo=12, hi=17, w=36, label=''):
        f = 0. if hi <= lo else max(0., min(1., (v - lo) / (hi - lo)))
        n = int(round(f * w))
        print(f'    {label:<22s} |{"█" * n}{"·" * (w - n)}| {v:.3f}')


R = Reporter


class CFG:
    seed = 42
    dataset = 'daniilantonov5/3d-mesh-quality-control'

    # ---- единственное разрешение: 336 ----
    # Нативный размер вида 512 px. 336 даёт 576 токенов против 256 при 224 и стоит
    # втрое дешевле, чем 448. По отдаче на час это лучшая точка.
    tile = 336
    view_mode = 'multiview'
    grid_rows, grid_cols = 2, 3

    backbone    = 'vit_base_patch14_reg4_dinov2.lvd142m'
    backbone_lr = 2e-5
    layer_decay = 0.70
    batch_size  = 2
    accum       = 12          # эффективный батч 24
    run_tag     = 'vitb336'

    drop_rate = 0.2
    # Лучшая эпоха во всех прошлых прогонах приходилась на 4-9 из 16, дальше метрика
    # падала и не восстанавливалась. 12 эпох — с запасом.
    epochs = 12
    mixup = 0.4
    asl_weight = 0.65         # пропорция BCE/ASL из прогона, давшего 14.987
    lr = 3e-4
    head_lr_mult = 10.0
    weight_decay = 0.05
    warmup_frac = 0.1
    label_smooth = 0.01
    amp = True
    num_workers = 4
    ema_decay = 0.999

    # ---- ML-Decoder ----
    use_mldecoder = True
    mld_tokens = 8
    mld_dim = 512
    mld_layers = 2

    # ---- симметрии стенда ----
    sym_aug = True
    azimuth_idx = [0, 1, 2, 3]
    pole_idx = [4, 5]
    view_dropout = 0.0        # создаёт условие `partial` при метке 0
    pixel_noise = 0.0         # противоречит классу `noisy` (вес 0.274)

    use_clip = True
    clip_model = ('ViT-B-32', 'laion2b_s34b_b79k')
    clip_pca = 64

    n_folds = 5
    folds_to_run = [0, 1, 2, 3, 4]

    n_snapshots = 3

    use_drive = True
    drive_dir = '/content/drive/MyDrive/sber_meshqc_v9'
    resume = True
    ckpt_every = 1
    ckpt_fp16 = True
    keep_fold_ckpt = False
    force_recompute = []

    # Готовый прогон v5 (14.987 на лидерборде) входит в ансамбль бесплатно:
    # его fold-предсказания весят 0.1 МБ и уже лежат на Drive.
    reuse_runs = {
        'v5_224': '/content/drive/MyDrive/sber_meshqc_v5/artifacts/dinos224_geo2_v5_fold{k}_preds.joblib',
    }

    artifact_cols = ['abstract','artifacts','intersection','lowpoly','noisy',
                     'open','partial','scale','set','simple']
    target_cols = artifact_cols + ['quality']
    n_targets = 11
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


CONFIG_VITB = dict(run_tag='vitb336', backbone='vit_base_patch14_reg4_dinov2.lvd142m',
                   tile=336, backbone_lr=2e-5, batch_size=2, accum=12, layer_decay=0.70)
CONFIG_EVA  = dict(run_tag='eva336', backbone='eva02_base_patch14_448.mim_in22k_ft_in22k_in1k',
                   tile=336, backbone_lr=2.5e-5, batch_size=2, accum=12, layer_decay=0.75)
CONFIGS = [CONFIG_VITB, CONFIG_EVA]


def seed_everything(seed=CFG.seed):
    random.seed(seed); np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


seed_everything()
IS_VIT = True


def apply_config(cfg):
    global IS_VIT
    for k, v in cfg.items():
        setattr(CFG, k, v)
    arch = CFG.backbone.split('.')[0]
    IS_VIT = any(arch.startswith(p) for p in ('vit_', 'eva', 'deit', 'beit'))
    if IS_VIT and CFG.tile % 14:
        CFG.tile = int(round(CFG.tile / 14) * 14)
        R.warn(f'tile округлён до кратного 14: {CFG.tile}')
    return CFG


RUN_T0 = time.time()
R.section('Конфигурация v9', 'только то, что подтверждено измерениями')
R.kv(**{'устройство': f'{CFG.device} '
                      f'({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})',
        'разрешение': f'{CFG.tile} ({(CFG.tile // 14) ** 2} токенов на вид)',
        'эпох x фолдов': f'{CFG.epochs} x {CFG.n_folds}',
        'loss': f'BCE {1 - CFG.asl_weight:.2f} + ASL {CFG.asl_weight:.2f}',
        'голова': 'ML-Decoder' if CFG.use_mldecoder else 'attention pooling',
        'переиспользуется': list(CFG.reuse_runs)})
for c in CONFIGS:
    R.kv(**{f"  {c['run_tag']}": f"{c['backbone'].split('.')[0]} @ {c['tile']}, "
                                 f"батч {c['batch_size']}x{c['accum']}"})

# =====================================================================================
#  СОСТАВ ФИНАЛЬНОГО АНСАМБЛЯ — ровно тот, что дал 15.541
#
#  | источник     | архитектура          | tile | OOF    | вес  | происхождение          |
#  |--------------|----------------------|------|--------|------|------------------------|
#  | vitb336      | DINOv2 ViT-B/14      | 336  | 14.451 | 0.75 | обучается этим кодом   |
#  | dino448_s2   | DINOv2 ViT-B/14      | 448  | 14.685 | 1.00 | импорт (§9.2)          |
#  | v5_224       | DINOv2 ViT-S/14      | 224  | 14.376 | 0.50 | импорт (§9.3)          |
#
#  Веса подобраны перебором по OOF, лидерборд в настройке не участвует.
#  Корреляция ошибок внутри тройки 0.77-0.84: источники сопоставимы по силе и
#  ошибаются в разных местах — именно это, а не мощность каждого, даёт прирост.
#  Разрешения 224 / 336 / 448 выбраны осознанно: нативный размер ракурса 512 px,
#  и модели на разных масштабах ловят разные дефекты.
# =====================================================================================
USE_CACHE = True      # True: готовые прогоны берутся с Drive. False: обучение с нуля.

CONFIG_VITB = dict(run_tag='vitb336', backbone='vit_base_patch14_reg4_dinov2.lvd142m',
                   tile=336, backbone_lr=2e-5, batch_size=2, accum=12, layer_decay=0.70)
CONFIG_448  = dict(run_tag='dino448_s2', backbone='vit_base_patch14_reg4_dinov2.lvd142m',
                   tile=448, backbone_lr=1.5e-5, batch_size=2, accum=12, layer_decay=0.70)
CONFIG_V5   = dict(run_tag='v5_224', backbone='vit_small_patch14_reg4_dinov2.lvd142m',
                   tile=224, backbone_lr=4e-5, batch_size=8, accum=3, layer_decay=0.75)
CONFIGS = [CONFIG_VITB, CONFIG_448, CONFIG_V5]
FINAL_TAGS = ['vitb336', 'dino448_s2', 'v5_224']

# Пути импорта для домашнего прогона. При USE_CACHE=False не используются:
# тогда все три конфигурации обучаются кодом ноутбука.
IMPORT_ZIP_GLOB = 'sber_meshqc_dino448_s2.zip'
IMPORT_V5_GLOB  = 'sber_meshqc_v5/artifacts/dinos224_geo2_v5_fold{k}_preds.joblib'

R.kv(**{'режим': 'импорт готовых прогонов' if USE_CACHE else 'полное обучение с нуля',
        'состав': ', '.join(FINAL_TAGS)})
if not USE_CACHE:
    R.warn(f'полное обучение: 3 модели x {CFG.n_folds} фолдов x {CFG.epochs} эпох '
           f'— ориентировочно 12-14 ч на A100')


## 1.1. Чекпоинты, кэш и возобновление

Colab рвёт сессию по таймауту и по лимиту GPU, а `/content` стирается **вместе с рантаймом**.
Поэтому весь прогресс уезжает на Google Drive, и каждый тяжёлый шаг сначала проверяет,
не посчитан ли он уже.

| артефакт | чем сохраняется | что даёт при обрыве |
|---|---|---|
| кэш рендеров (`img_*`) | PNG на диске | пересобираются только недостающие файлы |
| таблицы фич | parquet + чекпоинт раз в 1024 объекта | продолжает с последнего чанка |
| список фич после дедупликации | `joblib` | не пересчитывается |
| **состояние обучения фолда** | `torch.save`: веса, оптимизатор, scheduler, scaler, снапшоты, состояния ГСЧ | обучение продолжается **с той же эпохи**, а не с нуля |
| предсказания фолда (OOF + тест) | `joblib` | готовый фолд пропускается целиком |
| модели и предсказания GBM / стэкинга | `joblib` | не переобучаются |
| решающее правило | `joblib` | — |
| лог обучения по эпохам | дописываемый CSV | нужен для презентации (критерий «логи экспериментов») |

Запись **атомарная** (во временный файл, затем `os.replace`): обрыв прямо во время сохранения
не оставит битый чекпоинт. Состояния ГСЧ тоже в чекпоинте, поэтому продолженное обучение
идёт по той же траектории, что и непрерывное.

In [ ]:
import joblib, zipfile, time, shutil

# =====================================================================================
#  Работаем ЛОКАЛЬНО, Drive — зеркало.
#
#  Было: CFG.work = Drive, то есть каждая атомарная запись шла через FUSE-монтирование.
#  Чекпоинт содержит модель + EMA + 3 снапшота + состояние оптимизатора — порядка 0.5 ГБ,
#  и сохранялся КАЖДУЮ эпоху: 10 эпох x 5 фолдов x 2 прогона ≈ 50 ГБ записи на Drive за
#  один прогон ноутбука. Отсюда и обрывы сессии, и молча не долетавшие крупные артефакты
#  (в прошлой версии так потерялись gbm_geometry и dino_emb_train).
#
#  Стало: всё пишется в /content/work (локальный диск, мгновенно), а на Drive уходит
#  зеркало. Мелкие артефакты — сразу, чекпоинты — раз в MIRROR_CKPT_EVERY эпох и
#  обязательно в конце фолда, причём БЕЗ состояния оптимизатора: оно вдвое тяжелее самих
#  весов, а продолжать обучение код умеет и без него (см. try/except в train_fold).
# =====================================================================================
MIRROR_CKPT_EVERY = 3

LOCAL = Path('/content/work')
DRIVE = Path(CFG.drive_dir)
USE_DRIVE = False
if CFG.use_drive:
    try:
        from google.colab import drive as _gd
        _gd.mount('/content/drive')
        DRIVE.mkdir(parents=True, exist_ok=True)
        USE_DRIVE = True
    except Exception as e:
        print('Drive недоступен, прогресс не переживёт перезапуск:', e)

CFG.work  = LOCAL
CFG.cache = LOCAL / 'cache'
ART  = LOCAL / 'artifacts'
CKPT = LOCAL / 'ckpt'
for _d in (LOCAL, CFG.cache, ART, CKPT):
    _d.mkdir(parents=True, exist_ok=True)
LOG_CSV = LOCAL / 'training_log.csv'

R.section('Хранилище', f'работа: {LOCAL} | зеркало: {DRIVE if USE_DRIVE else "нет"}')


def pull_from_drive():
    """При старте тянем всё уже посчитанное с Drive на локальный диск."""
    if not USE_DRIVE:
        return
    n, mb = 0, 0.0
    for sub in ('artifacts', 'cache', 'ckpt'):
        s = DRIVE / sub
        if not s.exists():
            continue
        (LOCAL / sub).mkdir(parents=True, exist_ok=True)
        for p in s.iterdir():
            if not p.is_file() or p.suffix == '.tmp':
                continue
            q = LOCAL / sub / p.name
            if q.exists() and q.stat().st_size == p.stat().st_size:
                continue
            try:
                shutil.copy2(p, q); n += 1; mb += p.stat().st_size / 1e6
            except Exception as e:
                R.warn(f'{p.name} не скопировался: {type(e).__name__}')
    if (DRIVE / 'training_log.csv').exists() and not LOG_CSV.exists():
        shutil.copy2(DRIVE / 'training_log.csv', LOG_CSV)
    R.kv(**{'подтянуто с Drive': f'{n} файлов, {mb:.0f} МБ'})


pull_from_drive()


def push_to_drive(local_path):
    if not USE_DRIVE:
        return
    local_path = Path(local_path)
    try:
        rel = local_path.relative_to(LOCAL)
    except ValueError:
        return
    dst = DRIVE / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        tmp = dst.with_suffix(dst.suffix + '.tmp')
        shutil.copy2(local_path, tmp)
        os.replace(tmp, dst)
    except Exception as e:
        R.warn(f'зеркало {rel} не записалось ({type(e).__name__}) — локальная копия цела')


def _atomic(path, writer, mirror=True):
    """Пишем во временный файл рядом и подменяем: обрыв не испортит существующий артефакт."""
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    writer(tmp)
    os.replace(tmp, path)
    if mirror:
        push_to_drive(path)


def art_path(name):
    return ART / f'{name}.joblib'


def rt(name):
    """Имя артефакта, привязанное к конфигурации прогона (run_tag)."""
    return f'{CFG.run_tag}_{name}'


def has_artifact(name):
    return art_path(name).exists() and name not in CFG.force_recompute


def load_artifact(name):
    return joblib.load(art_path(name))


def save_artifact(name, obj, compress=3):
    _atomic(art_path(name), lambda p: joblib.dump(obj, p, compress=compress))
    return obj


def cached(name, fn, compress=3):
    """Единая точка «проверить существование -> иначе посчитать и сохранить»."""
    if CFG.resume and has_artifact(name):
        print(f'  [кэш] {name}: загружено')
        return load_artifact(name)
    t0 = time.time()
    obj = fn()
    save_artifact(name, obj, compress=compress)
    print(f'  [новое] {name}: посчитано за {time.time() - t0:.0f} c')
    return obj


# ---- состояния генераторов случайных чисел (для точного продолжения обучения) ----
def rng_state():
    st = {'py': random.getstate(), 'np': np.random.get_state(), 'torch': torch.get_rng_state()}
    if torch.cuda.is_available():
        st['cuda'] = torch.cuda.get_rng_state_all()
    return st


def set_rng_state(st):
    try:
        random.setstate(st['py']); np.random.set_state(st['np'])
        torch.set_rng_state(st['torch'].cpu() if torch.is_tensor(st['torch']) else st['torch'])
        if 'cuda' in st and torch.cuda.is_available():
            torch.cuda.set_rng_state_all([s.cpu() if torch.is_tensor(s) else s for s in st['cuda']])
    except Exception as e:
        print('  состояние ГСЧ не восстановлено (продолжаем с текущим):', e)


def _to_fp16(sd):
    return {k: (v.half() if torch.is_tensor(v) and v.is_floating_point() else v)
            for k, v in sd.items()}


def _to_fp32(sd):
    return {k: (v.float() if torch.is_tensor(v) and v.is_floating_point() else v)
            for k, v in sd.items()}


def save_ckpt(fold, payload):
    """Локально — полный чекпоинт каждую эпоху (это быстро). На Drive — облегчённый
    и по расписанию, иначе FUSE не справляется и роняет сессию."""
    p = CKPT / f'{CFG.run_tag}_fold{fold}.pt'
    _atomic(p, lambda q: torch.save(payload, q), mirror=False)
    ep = int(payload.get('epoch', 0)) + 1
    if USE_DRIVE and (ep % MIRROR_CKPT_EVERY == 0 or ep >= CFG.epochs):
        light = {k: v for k, v in payload.items() if k not in ('opt', 'sched', 'scaler')}
        lp = CKPT / f'{CFG.run_tag}_fold{fold}_light.pt'
        torch.save(light, lp)
        push_to_drive(lp)
        print(f'    [зеркало] чекпоинт эпохи {ep} -> Drive ({lp.stat().st_size / 1e6:.0f} МБ)')


def load_ckpt(fold):
    """Сначала полный локальный чекпоинт, затем облегчённый с зеркала."""
    if not CFG.resume:
        return None
    for nm in (f'{CFG.run_tag}_fold{fold}.pt', f'{CFG.run_tag}_fold{fold}_light.pt'):
        p = CKPT / nm
        if not p.exists():
            continue
        try:
            d = torch.load(p, map_location='cpu', weights_only=False)
            print(f'  чекпоинт {nm}: продолжаем с эпохи {d["epoch"] + 2}/{CFG.epochs}')
            return d
        except Exception as e:
            print(f'  чекпоинт {nm} не читается ({e})')
    return None


def drop_ckpt(fold):
    for nm in (f'{CFG.run_tag}_fold{fold}.pt', f'{CFG.run_tag}_fold{fold}_light.pt'):
        p = CKPT / nm
        if p.exists() and not CFG.keep_fold_ckpt:
            p.unlink()
        if USE_DRIVE:
            dp = DRIVE / 'ckpt' / nm
            if dp.exists() and not CFG.keep_fold_ckpt:
                dp.unlink()


def log_epoch(row):
    """Дописываемый лог: переживает обрывы, нужен для отчёта об экспериментах."""
    pd.DataFrame([row]).to_csv(LOG_CSV, mode='a', header=not LOG_CSV.exists(), index=False)
    if USE_DRIVE and int(row.get('epoch', 0)) % 2 == 0:
        push_to_drive(LOG_CSV)


done_folds = sorted(p.stem for p in ART.glob(f'{CFG.run_tag}_fold*_preds.joblib'))
R.kv(**{'посчитанные фолды': done_folds or 'нет',
        'незавершённые чекпоинты': [p.name for p in CKPT.glob('*.pt')] or 'нет'})


### 1.2. Аудит хранилища

Что уже посчитано и будет взято с диска, а что придётся считать заново. Запускайте перед каждым долгим прогоном.

In [ ]:
# ============ §1.2. Аудит хранилища: что уже посчитано и что можно удалить ============
R.section('Аудит хранилища', 'запускается до расчётов — показывает, что будет взято с диска')

rows = []
for base, nm in ((DRIVE, 'drive'), (LOCAL, 'local')):
    if not Path(base).exists():
        continue
    for p in Path(base).rglob('*'):
        if p.is_file():
            rows.append({'где': nm, 'путь': str(p.relative_to(base)),
                         'МБ': round(p.stat().st_size / 1e6, 2),
                         'изменён': time.strftime('%d.%m %H:%M', time.localtime(p.stat().st_mtime))})
FILES = pd.DataFrame(rows)
if len(FILES):
    R.kv(**{'файлов на Drive': int((FILES['где'] == 'drive').sum()),
            'файлов локально': int((FILES['где'] == 'local').sum()),
            'объём Drive': f"{FILES.loc[FILES['где'] == 'drive', 'МБ'].sum() / 1000:.2f} ГБ"})
    display(FILES.sort_values('МБ', ascending=False).head(30))
else:
    R.warn('хранилище пусто — всё будет считаться с нуля')

try:
    u = shutil.disk_usage(DRIVE if USE_DRIVE else LOCAL)
    R.kv(**{'свободно': f'{u.free / 1e9:.1f} ГБ'})
except Exception:
    pass

# ---- готовность каждого прогона ----
RUN_STATUS = {}
for c in CONFIGS:
    tag = c['run_tag']
    full = art_path(f'{tag}_cnn_preds')
    folds = sorted(ART.glob(f'{tag}_fold*_preds.joblib'))
    cks = sorted(CKPT.glob(f'{tag}_fold*.pt'))
    print(f'\nПРОГОН {tag}  ({c["backbone"]} @ {c["tile"]})')
    if full.exists():
        try:
            sc = load_artifact(f'{tag}_cnn_preds')['score']
            print(f'  ЗАВЕРШЁН ЦЕЛИКОМ, OOF {sc:.3f} — будет взят с диска за секунду')
            RUN_STATUS[tag] = 'done'
        except Exception as e:
            print(f'  файл {full.name} есть, но не читается ({type(e).__name__}) — прогон перезапустится')
            RUN_STATUS[tag] = 'broken'
    else:
        print(f'  готовых фолдов: {len(folds)}/{CFG.n_folds} -> '
              f'{[f.stem.split("fold")[1].split("_")[0] for f in folds] or "нет"}')
        RUN_STATUS[tag] = f'{len(folds)}/{CFG.n_folds}'
    for ck in cks:
        try:
            d = torch.load(ck, map_location='cpu', weights_only=False)
            print(f'  чекпоинт {ck.name}: эпоха {d["epoch"] + 1}/{CFG.epochs}, '
                  f'лучшая {d["best"]:.3f}, {ck.stat().st_size / 1e6:.0f} МБ — обучение продолжится отсюда')
        except Exception as e:
            print(f'  чекпоинт {ck.name}: БИТЫЙ ({type(e).__name__}) — фолд начнётся заново')

# ---- лог обучения ----
if LOG_CSV.exists():
    L = pd.read_csv(LOG_CSV)
    if {'fold', 'epoch', 'val_metric'} <= set(L.columns) and len(L):
        g = (L.drop_duplicates(['fold', 'epoch'], keep='last').groupby('fold')
             .agg(эпох=('epoch', 'count'), последняя=('epoch', 'max'), лучшая=('val_metric', 'max')))
        print(f'\nпройдено эпох всего: {len(L.drop_duplicates(["fold", "epoch"]))}')
        display(g.round(3))
else:
    print('\ntraining_log.csv нет — ни одна эпоха ещё не дописалась')

# ---- мусор ----
# FEAT_VER объявляется ниже, в §5, поэтому при первом проходе его ещё нет:
# тогда кэш геометрии просто не проверяем на устаревание.
_fv = globals().get('FEAT_VER')
JUNK = ([p for p in Path(DRIVE).rglob('*.tmp')] if USE_DRIVE else []) \
     + [p for p in LOCAL.rglob('*.tmp')] \
     + [p for p in CFG.cache.glob('*partial*')]
if _fv:
    JUNK += [p for p in CFG.cache.glob('mesh_*') if _fv not in p.name]
    if USE_DRIVE and (DRIVE / 'cache').exists():
        JUNK += [p for p in (DRIVE / 'cache').glob('mesh_*') if _fv not in p.name]
else:
    R.warn('FEAT_VER ещё не объявлен (§5) — устаревший кэш геометрии не проверяю. '
           'Чтобы проверить, выполните §5 и запустите эту ячейку ещё раз.')
JUNK = [p for p in JUNK if p.exists()]
print(f'\nмусора найдено: {len(JUNK)} файлов, {sum(p.stat().st_size for p in JUNK) / 1e9:.2f} ГБ')
for p in JUNK[:20]:
    print(f'    {p.name:<52s} {p.stat().st_size / 1e6:8.1f} МБ')
print('\n>>> Удалить его можно следующей ячейкой. Артефакты старых прогонов '
      '(dinos224_geo2_v5_*) НЕ считаются мусором: это готовое решение на 14.99, '
      'и они занимают доли мегабайта.')


## 2. Загрузка датасета с Kaggle

Структура на Kaggle: `train/train/{item_id}.png|.npz`, `test/test/...`, плюс `train.csv`, `test.csv`, `submission.csv`.
Пути ищем автопоиском, чтобы ноутбук не ломался при переименовании папок.

In [ ]:
import kagglehub

root = Path(kagglehub.dataset_download(CFG.dataset))
print('dataset root:', root)

def _pick_dir(mode):
    """Папка с максимальным числом .npz, в пути которой встречается train/test."""
    best, best_n = None, -1
    for p in root.rglob('*'):
        if not p.is_dir():
            continue
        if mode not in str(p).lower():
            continue
        n = sum(1 for _ in p.glob('*.npz'))
        if n > best_n:
            best, best_n = p, n
    if best is None or best_n == 0:
        raise FileNotFoundError(f'не найдена папка с .npz для {mode}')
    return best

def _pick_csv(mode):
    cands = list(root.rglob('*.csv'))
    def score(p):
        n = p.name.lower(); s = 0
        if mode == 'train':
            s += 10 * ('train' in n) - 20 * ('submission' in n) - 20 * ('test' in n)
        else:
            s += 10 * ('test' in n) - 5 * ('submission' in n) - 20 * ('train' in n)
        return s - 0.001 * len(n)
    return sorted(cands, key=score, reverse=True)[0]

TRAIN_DIR, TEST_DIR = _pick_dir('train'), _pick_dir('test')
TRAIN_CSV, TEST_CSV = _pick_csv('train'), _pick_csv('test')
print('train dir :', TRAIN_DIR, len(list(TRAIN_DIR.glob('*.npz'))), 'npz /',
      len(list(TRAIN_DIR.glob('*.png'))), 'png')
print('test  dir :', TEST_DIR,  len(list(TEST_DIR.glob('*.npz'))),  'npz /',
      len(list(TEST_DIR.glob('*.png'))),  'png')
print('train csv :', TRAIN_CSV)
print('test  csv :', TEST_CSV)

train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)[['item_id']].copy()
train_df['item_id'] = train_df['item_id'].astype(str)
test_df['item_id']  = test_df['item_id'].astype(str)

if 'quality' not in train_df.columns:
    train_df['quality'] = (train_df[CFG.artifact_cols].sum(1) == 0).astype(int)

print(train_df.shape, test_df.shape)
train_df.head()

In [ ]:
from sklearn.metrics import f1_score

def competition_metric(y_true, y_pred):
    """10*F1(quality) + 10*F1_weighted(10 дефектов) — точная формула организаторов."""
    f1_q = f1_score(y_true[:, 10], y_pred[:, 10], zero_division=0)
    f1_a = f1_score(y_true[:, :10], y_pred[:, :10], average='weighted', zero_division=0)
    return 10 * f1_q + 10 * f1_a, f1_q, f1_a

# сколько стоит "OR-правило" quality = нет дефектов, если по дефектам ошибаться независимо
Y = train_df[CFG.target_cols].values
for err in [0.02, 0.05, 0.10]:
    rng = np.random.default_rng(0)
    fake = Y.copy()
    flip = rng.random(fake[:, :10].shape) < err
    fake[:, :10] = np.abs(fake[:, :10] - flip)
    fake[:, 10] = (fake[:, :10].sum(1) == 0).astype(int)
    s, fq, fa = competition_metric(Y, fake)
    print(f'FPR/FNR по дефектам {err:.0%} -> метрика {s:5.2f} (quality F1={fq:.3f}, artefacts F1={fa:.3f})')
print('\n>>> Ошибка 5% по дефектам уже срезает quality F1 до ~0.7. '
      'Поэтому quality предсказывается ОТДЕЛЬНОЙ головой, а не только правилом.')

## 4. Рендеры: разрезаем мозаику на 6 видов и кэшируем

В `{item_id}.png` лежат 6 видов одним листом (4 азимута с шагом 90° + top + bottom).
Раскладку определяем по соотношению сторон (поддержаны 3×2, 2×3, 6×1, 1×6), поэтому
ноутбук не сломается, если у части файлов сетка транспонирована.

Кэш: каждая картинка ужимается так, чтобы один вид был `CFG.tile` пикселей, и складывается
обратно в мозаику — один PNG на объект (~150 КБ), декодирование в DataLoader-е дешёвое.

In [ ]:
from PIL import Image
from concurrent.futures import ProcessPoolExecutor

Image.MAX_IMAGE_PIXELS = None

def infer_grid(w, h):
    """(rows, cols) для 6 видов по соотношению сторон."""
    ar = w / h
    cands = {(2, 3): 3 / 2, (3, 2): 2 / 3, (1, 6): 6.0, (6, 1): 1 / 6}
    return min(cands, key=lambda k: abs(math.log(ar / cands[k])))

def split_views(img):
    """PIL.Image -> список из 6 PIL.Image в каноническом порядке чтения."""
    w, h = img.size
    r, c = infer_grid(w, h)
    tw, th = w // c, h // r
    return [img.crop((j * tw, i * th, (j + 1) * tw, (i + 1) * th)) for i in range(r) for j in range(c)]

def make_cache(args):
    src, dst, tile = args
    try:
        img = Image.open(src).convert('RGB')
        views = [v.resize((tile, tile), Image.BILINEAR) for v in split_views(img)]
        out = Image.new('RGB', (3 * tile, 2 * tile))
        for k, v in enumerate(views):
            out.paste(v, ((k % 3) * tile, (k // 3) * tile))
        out.save(dst, format='PNG', optimize=False, compress_level=1)
        return 1
    except Exception:
        return 0

def build_image_cache(ids, src_dir, split):
    out_dir = CFG.cache / f'img_{split}_{CFG.tile}'
    out_dir.mkdir(parents=True, exist_ok=True)
    jobs = [(str(Path(src_dir) / f'{i}.png'), str(out_dir / f'{i}.png'), CFG.tile)
            for i in ids if not (out_dir / f'{i}.png').exists()
            and (Path(src_dir) / f'{i}.png').exists()]
    if jobs:
        with ProcessPoolExecutor(max_workers=os.cpu_count()) as ex:
            ok = list(tqdm(ex.map(make_cache, jobs, chunksize=32), total=len(jobs),
                           desc=f'cache {split}'))
        print(f'{split}: закэшировано {sum(ok)}/{len(jobs)}')
    missing = [i for i in ids if not (out_dir / f'{i}.png').exists()]
    print(f'{split}: всего в кэше {len(ids) - len(missing)}/{len(ids)}, нет картинки у {len(missing)}')
    return out_dir, set(missing)

# --- сколько пикселей на вид даёт исходный рендер: апскейл выше этого бессмысленно ---
_probe = [Image.open(Path(TRAIN_DIR) / f'{i}.png').size
          for i in train_df['item_id'].head(40) if (Path(TRAIN_DIR) / f'{i}.png').exists()]
if _probe:
    _sizes = pd.Series([f'{w}x{h}' for w, h in _probe]).value_counts()
    print('исходные размеры листа с 6 видами:'); print(_sizes.head(5).to_string())
    _w, _h = _probe[0]
    _r, _c = infer_grid(_w, _h)
    NATIVE_TILE = int(min(_w // _c, _h // _r))
    R.kv(**{'раскладка': f'{_r}x{_c}', 'нативный размер вида': f'{NATIVE_TILE} px',
            'используем': f'{CFG.tile} ({100 * (CFG.tile / NATIVE_TILE) ** 2:.0f}% площади кадра)'})
    if CFG.tile > NATIVE_TILE:
        R.warn(f'tile {CFG.tile} > нативного — это апскейл, снижаю')
        CFG.tile = NATIVE_TILE

IMG_TRAIN, MISS_TRAIN = build_image_cache(train_df['item_id'].tolist(), TRAIN_DIR, 'train')
IMG_TEST,  MISS_TEST  = build_image_cache(test_df['item_id'].tolist(),  TEST_DIR,  'test')

## 5. Геометрические фичи из `.npz` — **со сваркой вершин**

### Что было сломано в прошлом прогоне

В диагностике «топ-фича на класс» не оказалось **ни одной** топологической фичи:
`open` объяснялся картиночной `pxstd_min`, `set` — `bbox_fill_view`, `lowpoly` — `uniq_normals`.
Это подпись **несваренных вершин**: экспортёры (glTF/OBJ с раздельными нормалями и UV)
дублируют вершину на каждую грань, поэтому в сыром массиве индексов
**ни одно ребро не принадлежит двум граням**. Следствие:

* двугранных углов не существует вовсе (`dihed_*` тождественно 0);
* `boundary_edge_ratio = 1.0` у **любого** меша, включая замкнутый;
* `n_comp` = числу треугольников, `watertight = 0` всегда.

То есть вся топологическая ветка была мертва, а `open`/`artifacts` (F1 ≈ 0.30) вытягивались
случайными прокси. Ниже — сварка вершин по квантованной сетке `1e-6` диагонали bbox
(упаковка трёх координат в один int64, без коллизий), удаление вырожденных и дублирующихся
граней, и только потом топология.

### Новые фичи и их мишени

| фича | класс | обоснование |
|---|---|---|
| `boundary_edge_ratio`, `n_boundary_loops`, `watertight`, `boundary_vert_frac` | `open` | «полые внутри либо состоят из плоских поверхностей» = незамкнутая поверхность с граничными петлями |
| `nonmanifold_edge_ratio`, `dup_face_ratio`, `degen_after_weld`, `vol_ratio` | `artifacts` | non-manifold рёбра, дубли граней и несогласованная намотка = «объект выглядит сломанным» |
| `solidity` (V/V выпуклой оболочки), `sphericity`, `normal_top6_mass` | `simple` | у примитивов вся площадь лежит в нескольких кластерах нормалей, solidity → 1 |
| `dihedf_*` (folded углы), `dihed_entropy`, `smoothness`, `valence_*` | `lowpoly` / `noisy` | фасетчатость регулярна (низкая энтропия), скан шумит хаотично (высокая) |
| угловой дефект `defect_*` (гауссова кривизна) | `noisy`, `lowpoly` | точечная кривизна: у гладкой поверхности ≈ 0, у шума разброс |
| `n_comp`, `comp_top1`, `comp_entropy` | `set` | «объект разделён в пространстве» — теперь считается по сваренному мешу |
| `bbox_iou_max`, `inside_frac_max` | `intersection` | доля вершин одной компоненты внутри bbox другой |

Проверка на синтетике (несваренный куб/сфера дают ровно те же значения, что сваренные):
гладкая сфера `dihedf=3.0°`, lowpoly-40 `22.9°`, шумный скан `13.6°`;
`normal_top6_mass`: куб `1.00`, сфера `0.02`; `solidity`: куб `1.00`, плоскость `0.00`;
сфера с вырезанной шапкой — `watertight=0`, `n_boundary_loops=1`.

In [ ]:
import os, gc, math, numpy as np, pandas as pd
from pathlib import Path
from PIL import Image
from scipy import ndimage
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from scipy.spatial import ConvexHull, QhullError
from concurrent.futures import ProcessPoolExecutor
from concurrent.futures.process import BrokenProcessPool
import multiprocessing as mp

FACE_CAP  = 300_000
VERT_CAP  = 200_000
HULL_CAP  = 20_000
COMP_CAP  = 3_000_000
WELD_TOL  = 1e-6         # доля диагонали bbox
FEAT_VER  = 'v4selfint'  # версия набора фич: входит в имя кэша. Поднята с v3weld, т.к.
                         # добавлены §4.5.1-4.5.3 (самопересечения/качество треугольников/
                         # non-manifold вершины) — старый parquet без них не подхватится молча
# Было min(4, ...) — на машине с 8-12 ядрами это втрое резало самый дорогой шаг.
N_WORKERS = max(1, (os.cpu_count() or 4) - 1)
CHUNK     = 512          # чаще чекпоинт: при обрыве теряется меньше

def _pack21(q):
    """Три целых из [0, 2**21) -> один int64. Точно, без коллизий."""
    return (q[:, 0] << 42) | (q[:, 1] << 21) | q[:, 2]

def _hash3(q):
    q = q.astype(np.int64, copy=False)
    return (q[:, 0] * 73856093) ^ (q[:, 1] * 19349663) ^ (q[:, 2] * 83492791)

def _stats(x, pref):
    keys = ['mean', 'std', 'p10', 'p50', 'p90', 'max', 'cv']
    if len(x) == 0:
        return {f'{pref}_{k}': 0.0 for k in keys}
    x = np.asarray(x, dtype=np.float32)
    m, s = float(x.mean()), float(x.std())
    q10, q50, q90 = np.percentile(x, [10, 50, 90])
    return {f'{pref}_mean': m, f'{pref}_std': s, f'{pref}_p10': float(q10),
            f'{pref}_p50': float(q50), f'{pref}_p90': float(q90),
            f'{pref}_max': float(x.max()), f'{pref}_cv': float(s / (abs(m) + 1e-9))}

def _edge_table(Fi, nfw):
    """Рёбра -> (номер группы, число граней на ребро, отсортированные id граней)."""
    E = np.concatenate([Fi[:, [0, 1]], Fi[:, [1, 2]], Fi[:, [2, 0]]], 0)
    E.sort(axis=1)
    fid = np.tile(np.arange(nfw, dtype=np.int32), 3)
    order = np.lexsort((E[:, 1], E[:, 0]))
    Es, fids = E[order], fid[order]
    new = np.ones(len(Es), dtype=bool)
    new[1:] = (Es[1:] != Es[:-1]).any(1)
    grp = np.cumsum(new) - 1
    counts = np.bincount(grp)
    return E, Es, fids, grp, counts, new

def mesh_features(npz_path):
    f = {'geo_ok': 0.0, 'geo_subsampled': 0.0}
    try:
        f['npz_mb'] = os.path.getsize(npz_path) / 1e6
    except Exception:
        return f
    try:
        with np.load(npz_path, allow_pickle=False) as d:
            keys = list(d.keys())
            V = np.asarray(d['vertices' if 'vertices' in keys else keys[0]], dtype=np.float32)
            Fc = None
            for k in ('faces', 'triangles', 'f'):
                if k in keys:
                    Fc = np.asarray(d[k], dtype=np.int64); break
            if Fc is None and len(keys) > 1:
                Fc = np.asarray(d[keys[1]], dtype=np.int64)
    except Exception:
        return f
    if V.ndim != 2 or V.shape[1] != 3 or len(V) == 0:
        return f
    if Fc is None or Fc.ndim != 2 or Fc.shape[1] != 3 or len(Fc) == 0:
        Fc = np.zeros((0, 3), dtype=np.int64)
    else:
        Fc = Fc[(Fc >= 0).all(1) & (Fc < len(V)).all(1)]

    f['geo_ok'] = 1.0
    nv_raw, nf_raw = len(V), len(Fc)
    vmin, vmax = V.min(0), V.max(0)
    ext = (vmax - vmin).astype(np.float64)
    ext_s = np.sort(ext)[::-1]
    diag = float(np.linalg.norm(ext)) + 1e-12
    center = ((vmin + vmax) / 2).astype(np.float32)
    inv_s = np.float32(1.0 / diag)
    f.update({'n_verts_raw': float(nv_raw), 'n_faces': float(nf_raw),
              'log_verts': math.log1p(nv_raw), 'log_faces': math.log1p(nf_raw),
              'ext_max': float(ext_s[0]), 'ext_mid': float(ext_s[1]), 'ext_min': float(ext_s[2]),
              'ext_ratio_min': float(ext_s[2] / (ext_s[0] + 1e-12)),
              'ext_ratio_mid': float(ext_s[1] / (ext_s[0] + 1e-12)),
              'bbox_diag': diag,
              'bbox_fill': float(nv_raw / (ext.prod() + 1e-12)) if ext.prod() > 0 else 0.0})

    # ================== СВАРКА ВЕРШИН ==================
    # Экспортёры (glTF/OBJ с раздельными нормалями и UV) дублируют вершины на каждую грань.
    # Без сварки НИ ОДНО ребро не имеет двух граней: двугранные углы пусты, boundary=1.0,
    # компонент столько же, сколько треугольников. Вся топология обязана считаться после сварки.
    step = np.float32(diag * WELD_TOL)
    q = np.clip(np.rint((V - vmin) / step), 0, (1 << 21) - 1).astype(np.int64)
    code = _pack21(q)
    del q
    ucode, first, invmap = np.unique(code, return_index=True, return_inverse=True)
    del code, ucode
    nv = len(first)
    f['n_verts'] = float(nv)
    f['weld_ratio'] = float(1.0 - nv / max(nv_raw, 1))          # 0 = уже сварен, ~0.83 = по 3 вершины на грань
    f['f_per_v'] = nf_raw / max(nv, 1)
    Vw_full = V[first]
    del first

    if nf_raw:
        Fw = invmap[Fc]
        ok = (Fw[:, 0] != Fw[:, 1]) & (Fw[:, 1] != Fw[:, 2]) & (Fw[:, 0] != Fw[:, 2])
        f['degen_after_weld'] = float(1.0 - ok.mean())
        Fw = Fw[ok]
        hf = _hash3(np.sort(Fw, axis=1))
        _, uidx = np.unique(hf, return_index=True)
        f['dup_face_ratio'] = float(1.0 - len(uidx) / max(len(Fw), 1))
        Fw = Fw[np.sort(uidx)]
        del hf, uidx, ok
    else:
        Fw = np.zeros((0, 3), dtype=np.int64)
        f['degen_after_weld'] = 0.0; f['dup_face_ratio'] = 0.0
    del invmap, Fc
    nf = len(Fw)

    # ---- вершинные статистики (подвыборка) ----
    Vs = Vw_full[::max(1, int(np.ceil(nv / VERT_CAP)))]
    if nv > VERT_CAP:
        f['geo_subsampled'] = 1.0
    Vs = (Vs - center) * inv_s
    try:
        if len(Vs) < 3:
            raise ValueError('too few vertices')
        ev = np.clip(np.linalg.eigvalsh(np.cov(Vs.T.astype(np.float64)))[::-1], 1e-16, None)
        f.update({'pca_1': float(ev[0]), 'pca_2': float(ev[1]), 'pca_3': float(ev[2]),
                  'pca_flat': float(ev[2] / ev[0]), 'pca_lin': float(ev[1] / ev[0]),
                  'pca_aniso': float((ev[0] - ev[2]) / ev.sum())})
    except Exception:
        f.update({k: 0.0 for k in ['pca_1', 'pca_2', 'pca_3', 'pca_flat', 'pca_lin', 'pca_aniso']})

    # ---- выпуклая оболочка: solidity/сферичность -> simple, open ----
    f.update({'hull_ok': 0.0, 'solidity': 0.0, 'solidity_signed': 0.0, 'hull_area_ratio': 0.0,
              'hull_pts_frac': 0.0, 'sphericity': 0.0, 'hull_vol': 0.0, 'hull_area': 0.0})
    try:
        Vh = Vs[::max(1, int(np.ceil(len(Vs) / HULL_CAP)))].astype(np.float64)
        if len(Vh) >= 8:
            hull = ConvexHull(Vh, qhull_options='QJ')
            f['hull_ok'] = 1.0
            f['hull_vol'] = float(hull.volume)
            f['hull_pts_frac'] = len(hull.vertices) / len(Vh)
            f['hull_area'] = float(hull.area)
    except (QhullError, ValueError, MemoryError):
        pass

    if nf == 0:
        f['no_faces'] = 1.0
        f.update({'area_total': 0.0, 'volume': 0.0, 'abs_volume': 0.0, 'vol_ratio': 0.0,
                  'vol_over_area': 0.0, 'absvol_over_area': 0.0, 'vol_over_bbox': 0.0,
                  'absvol_over_bbox': 0.0})
        for pref in ('dihed', 'dihedf', 'farea', 'elen', 'aspect', 'valence', 'defect'):
            f.update(_stats([], pref))
        return f
    f['no_faces'] = 0.0

    # ---- точные площадь/объём по полному сваренному мешу ----
    total_area, vol6, absvol6 = 0.0, 0.0, 0.0
    for s0 in range(0, nf, 500_000):
        Pc = ((Vw_full[Fw[s0:s0 + 500_000]] - center) * inv_s).astype(np.float64)
        cr_c = np.cross(Pc[:, 1] - Pc[:, 0], Pc[:, 2] - Pc[:, 0])
        total_area += 0.5 * float(np.linalg.norm(cr_c, axis=1).sum())
        contrib = (Pc[:, 0] * np.cross(Pc[:, 1], Pc[:, 2])).sum(1)
        vol6 += float(contrib.sum()); absvol6 += float(np.abs(contrib).sum())
        del Pc, cr_c, contrib
    total_area += 1e-12
    vol = abs(vol6 / 6.0)
    absvol = absvol6 / 6.0
    # vol_ratio ~1 у корректно ориентированного меша и ~0 при несогласованной намотке граней
    # (сама по себе сильная улика для artifacts/noisy). Объём для solidity берём устойчивый.
    f.update({'area_total': total_area, 'volume': vol, 'abs_volume': absvol,
              'vol_ratio': vol / (absvol + 1e-12),
              'vol_over_area': vol / (total_area ** 1.5 + 1e-12),
              'absvol_over_area': absvol / (total_area ** 1.5 + 1e-12),
              'vol_over_bbox': vol / (float(np.prod(ext * inv_s)) + 1e-12),
              'absvol_over_bbox': absvol / (float(np.prod(ext * inv_s)) + 1e-12)})
    if f['hull_ok']:
        f['solidity'] = absvol / (f['hull_vol'] + 1e-12)
        f['solidity_signed'] = vol / (f['hull_vol'] + 1e-12)
        f['hull_area_ratio'] = total_area / (f.get('hull_area', 0.0) + 1e-12)
        f['sphericity'] = (math.pi ** (1 / 3)) * (6 * absvol) ** (2 / 3) / (total_area + 1e-12)

    # ---- компоненты связности на полном СВАРЕННОМ меше ----
    f.update({'n_comp': 1.0, 'n_comp_big': 1.0, 'comp_top1': 1.0, 'comp_top2': 0.0,
              'comp_entropy': 0.0, 'bbox_iou_max': 0.0, 'bbox_iou_mean': 0.0,
              'bbox_iou_frac_pos': 0.0, 'comp_sep_max': 0.0, 'comp_sep_mean': 0.0,
              'inside_frac_max': 0.0, 'comp_exact': 1.0})
    if nf <= COMP_CAP:
        try:
            Ef = np.concatenate([Fw[:, [0, 1]], Fw[:, [1, 2]], Fw[:, [2, 0]]], 0).astype(np.int32)
            A = coo_matrix((np.ones(len(Ef), dtype=np.int8), (Ef[:, 0], Ef[:, 1])), shape=(nv, nv))
            del Ef
            ncomp, lab = connected_components(A, directed=False)
            del A
            used = np.unique(Fw)
            sizes_all = np.bincount(lab[used], minlength=ncomp).astype(np.float64)
            sizes = np.sort(sizes_all[sizes_all > 0])[::-1]
            tot = sizes.sum()
            big_idx = np.where(sizes_all > 0.005 * tot)[0]
            f.update({'n_comp': float(len(sizes)), 'n_comp_big': float(len(big_idx)),
                      'comp_top1': float(sizes[0] / tot),
                      'comp_top2': float(sizes[1] / tot) if len(sizes) > 1 else 0.0,
                      'comp_entropy': float(-((sizes / tot) * np.log(sizes / tot + 1e-12)).sum())})
            if len(big_idx) > 1:
                top = big_idx[np.argsort(-sizes_all[big_idx])][:8]
                pts_l, boxes = [], []
                for c in top:
                    ic = used[lab[used] == c]
                    if len(ic) > 20_000:
                        ic = ic[::len(ic) // 20_000 + 1]
                    p = (Vw_full[ic] - center) * inv_s
                    pts_l.append(p); boxes.append((p.min(0), p.max(0)))
                ious, seps, insides = [], [], []
                for i in range(len(boxes)):
                    for j in range(len(boxes)):
                        if i == j:
                            continue
                        lo, hi = boxes[j]
                        insides.append(float(((pts_l[i] >= lo) & (pts_l[i] <= hi)).all(1).mean()))
                        if j <= i:
                            continue
                        lo2 = np.maximum(boxes[i][0], boxes[j][0])
                        hi2 = np.minimum(boxes[i][1], boxes[j][1])
                        it = float(np.prod(np.clip(hi2 - lo2, 0, None)))
                        vi = float(np.prod(np.clip(boxes[i][1] - boxes[i][0], 1e-6, None)))
                        vj = float(np.prod(np.clip(boxes[j][1] - boxes[j][0], 1e-6, None)))
                        ious.append(it / (vi + vj - it + 1e-12))
                        seps.append(float(np.linalg.norm(
                            (boxes[i][0] + boxes[i][1]) / 2 - (boxes[j][0] + boxes[j][1]) / 2)))
                f.update({'bbox_iou_max': float(max(ious)), 'bbox_iou_mean': float(np.mean(ious)),
                          'bbox_iou_frac_pos': float(np.mean(np.asarray(ious) > 1e-6)),
                          'comp_sep_max': float(max(seps)), 'comp_sep_mean': float(np.mean(seps)),
                          'inside_frac_max': float(max(insides))})
            del lab, sizes_all, sizes, used
        except Exception:
            f['comp_exact'] = 0.0
    else:
        f['comp_exact'] = 0.0

    # ---- пространственный патч для рёберной топологии ----
    if nf > FACE_CAP:
        f['geo_subsampled'] = 1.0
        anchor = (Vw_full[Fw[:, 0]] - vmin) / (ext.astype(np.float32) + 1e-9)
        Fp, g = None, 1
        while g < 40:
            g += 1
            cid = np.minimum((anchor * g).astype(np.int32), g - 1)
            key = (cid[:, 0] * g + cid[:, 1]) * g + cid[:, 2]
            cnt = np.bincount(key, minlength=g ** 3)
            best = int(np.argmax(cnt))
            if cnt[best] <= FACE_CAP:
                Fp = Fw[key == best]; break
        if Fp is None or len(Fp) < 1000:
            Fp = Fw[:FACE_CAP]
        f['patch_grid'] = float(g)
        del anchor
    else:
        Fp = Fw
        f['patch_grid'] = 1.0
    f['patch_face_frac'] = len(Fp) / max(nf, 1)
    del Fw
    nfw = len(Fp)

    uvp, Fi = np.unique(Fp, return_inverse=True)
    Fi = Fi.reshape(nfw, 3).astype(np.int32)
    Vp = ((Vw_full[uvp] - center) * inv_s).astype(np.float32)
    nvp = len(uvp)
    del uvp, Fp, Vw_full, V

    P = Vp[Fi]
    cr = np.cross(P[:, 1] - P[:, 0], P[:, 2] - P[:, 0])
    area2 = np.linalg.norm(cr, axis=1)
    area = 0.5 * area2
    nrm = cr / (area2[:, None] + 1e-12)
    del cr
    f.update(_stats(area / (area.mean() + 1e-12), 'farea'))
    f['degenerate_ratio'] = float((area < 1e-10).mean())

    L = np.stack([np.linalg.norm(P[:, 1] - P[:, 0], axis=1),
                  np.linalg.norm(P[:, 2] - P[:, 1], axis=1),
                  np.linalg.norm(P[:, 0] - P[:, 2], axis=1)], 1)
    f.update(_stats(L.ravel() / (L.mean() + 1e-12), 'elen'))
    aspect = L.max(1) / (L.min(1) + 1e-12)
    f.update(_stats(np.log1p(aspect), 'aspect'))
    f['sliver_ratio'] = float((aspect > 20).mean())

    # ---- §4.5.2 качество треугольников: худший угол + нормализованная форма ----
    # min-угол — классический индикатор "плохого" элемента (aspect ratio его не всегда
    # ловит: длинный тупоугольный треугольник может иметь умеренный aspect, но острый
    # min-угол). tri_quality = 4*sqrt(3)*area/(l0^2+l1^2+l2^2) -> 1.0 для равностороннего,
    # ->0 для вырожденного; оба считаются векторно из уже посчитанных L и area, без
    # дополнительного прохода по мешу.
    L0, L1, L2 = L[:, 0], L[:, 1], L[:, 2]
    with np.errstate(divide='ignore', invalid='ignore'):
        cA = np.clip((L1 ** 2 + L2 ** 2 - L0 ** 2) / (2 * L1 * L2 + 1e-12), -1, 1)
        cB = np.clip((L0 ** 2 + L2 ** 2 - L1 ** 2) / (2 * L0 * L2 + 1e-12), -1, 1)
        cC = np.clip((L0 ** 2 + L1 ** 2 - L2 ** 2) / (2 * L0 * L1 + 1e-12), -1, 1)
    tri_ang = np.degrees(np.arccos(np.stack([cA, cB, cC], 1)))
    min_ang, max_ang = tri_ang.min(1), tri_ang.max(1)
    f.update(_stats(min_ang, 'minang')); f.update(_stats(max_ang, 'maxang'))
    f['sliver_angle_ratio'] = float((min_ang < 5).mean())
    tri_quality = np.clip((4 * math.sqrt(3) * area) / (L0 ** 2 + L1 ** 2 + L2 ** 2 + 1e-12), 0, 1)
    f.update(_stats(tri_quality, 'triq'))
    del L0, L1, L2, cA, cB, cC, tri_ang, min_ang, max_ang, tri_quality

    for qd in (1, 2):
        f[f'uniq_normals_q{qd}'] = len(np.unique(_hash3(np.rint(nrm * 10 ** qd)))) / nfw
    # концентрация нормалей: доля площади в 6 крупнейших кластерах -> примитивы
    hn = _hash3(np.rint(nrm * 20))
    _, ninv = np.unique(hn, return_inverse=True)
    mass = np.bincount(ninv, weights=area)
    mass = np.sort(mass)[::-1]
    f['normal_top6_mass'] = float(mass[:6].sum() / (mass.sum() + 1e-12))
    f['normal_top1_mass'] = float(mass[0] / (mass.sum() + 1e-12))
    del hn, ninv, mass

    # ---- §4.5.1 самопересечения: близкие по центроиду грани без общей вершины ----
    # intersection сейчас ловится только CNN с рендеров; хуже того, его confusion
    # портит quality (10 из 16 баллов), даже когда сам класс почти ничего не весит
    # (см. §14, "потолок по классам": intersection+scale = 0.18 балла при идеальном
    # предсказании) — поэтому цель этих фич не "поднять F1(intersection)" сама по себе,
    # а снизить количество ложных срабатываний, которые размывают quality.
    # cKDTree.query_pairs даёт кандидатов почти даром; точный тест — 6 рёберно-плоскостных
    # проверок на кандидата (Möller-style), полностью векторизован по всем парам разом.
    f.update({'selfint_ok': 0.0, 'selfint_pair_ratio': 0.0, 'selfint_face_ratio': 0.0,
              'selfint_any': 0.0, 'selfint_capped': 0.0})
    # 40k вместо 120k: самопересечения нужны для intersection, который стоит
    # 0.115 балла ДАЖЕ при идеальном предсказании — точность там не окупает времени.
    SELFINT_FACE_CAP = 40_000
    SELFINT_PAIR_CAP = 400_000
    if 0 < nfw <= SELFINT_FACE_CAP:
        try:
            from scipy.spatial import cKDTree
            cen = Vp[Fi].mean(1)
            elen_med = float(np.median(L)) if len(L) else 1e-6
            r = max(elen_med * 2.5, 1e-6)
            pairs = cKDTree(cen).query_pairs(r=r, output_type='ndarray')
            if len(pairs):
                share = (Fi[pairs[:, 0], :, None] == Fi[pairs[:, 1], None, :]).any((1, 2))
                pairs = pairs[~share]
            if len(pairs) > SELFINT_PAIR_CAP:
                sel = np.random.default_rng(0).choice(len(pairs), SELFINT_PAIR_CAP, replace=False)
                pairs = pairs[sel]
                f['selfint_capped'] = 1.0
            f['selfint_ok'] = 1.0
            if len(pairs):
                ii, jj = pairs[:, 0], pairs[:, 1]
                PA, PB = Vp[Fi[ii]], Vp[Fi[jj]]
                nA, nB = nrm[ii], nrm[jj]

                def _edge_hits(a, b, p0, p1, p2, n):
                    d = np.einsum('ij,ij->i', n, p0)
                    ta = np.einsum('ij,ij->i', n, a) - d
                    tb = np.einsum('ij,ij->i', n, b) - d
                    denom = ta - tb
                    safe = np.abs(denom) > 1e-12
                    t = np.clip(np.divide(ta, denom, out=np.zeros_like(ta), where=safe), 0.0, 1.0)
                    pt = a + t[:, None] * (b - a)
                    e0 = np.cross(p1 - p0, pt - p0); e1 = np.cross(p2 - p1, pt - p1)
                    e2 = np.cross(p0 - p2, pt - p2)
                    s0 = np.einsum('ij,ij->i', e0, n); s1 = np.einsum('ij,ij->i', e1, n)
                    s2 = np.einsum('ij,ij->i', e2, n)
                    inside = (((s0 >= -1e-9) & (s1 >= -1e-9) & (s2 >= -1e-9)) |
                              ((s0 <= 1e-9) & (s1 <= 1e-9) & (s2 <= 1e-9)))
                    return (ta * tb <= 0) & safe & inside

                hit = np.zeros(len(pairs), dtype=bool)
                for e in range(3):
                    hit |= _edge_hits(PA[:, e], PA[:, (e + 1) % 3], PB[:, 0], PB[:, 1], PB[:, 2], nB)
                for e in range(3):
                    hit |= _edge_hits(PB[:, e], PB[:, (e + 1) % 3], PA[:, 0], PA[:, 1], PA[:, 2], nA)
                n_hit = int(hit.sum())
                f['selfint_pair_ratio'] = float(n_hit / max(nfw, 1))
                f['selfint_any'] = float(n_hit > 0)
                if n_hit:
                    f['selfint_face_ratio'] = float(
                        len(np.unique(np.concatenate([ii[hit], jj[hit]]))) / max(nfw, 1))
                del PA, PB, nA, nB, hit
            del cen, pairs
        except Exception:
            pass
    else:
        f['selfint_capped'] = 1.0

    # ---- рёбра/углы ПОСЛЕ сварки + диагностика "как было бы БЕЗ сварки" ----
    E, Es, fids, grp, counts, _ = _edge_table(Fi, nfw)
    n_edges = len(counts)
    f.update({'n_edges': float(n_edges),
              'boundary_edge_ratio': float((counts == 1).mean()),
              'nonmanifold_edge_ratio': float((counts >= 3).mean()),
              'euler_norm': float((nvp - n_edges + nfw) / max(nfw, 1)),
              'euler_char': float(nvp - n_edges + nfw),
              'watertight': float((counts == 1).sum() == 0 and (counts >= 3).sum() == 0),
              'manifold_edge_ratio': float((counts == 2).mean())})

    # число граничных петель -> сколько «дыр» в поверхности (open)
    bnd = np.where(counts == 1)[0]
    f['nonmanifold_vert_ratio'] = 0.0
    if len(bnd):
        st = np.searchsorted(grp, bnd)
        be = Es[st]
        try:
            Ab = coo_matrix((np.ones(len(be), dtype=np.int8), (be[:, 0], be[:, 1])), shape=(nvp, nvp))
            nb_comp, lb = connected_components(Ab, directed=False)
            used_b = np.unique(be)
            f['n_boundary_loops'] = float(len(np.unique(lb[used_b])))
            f['boundary_vert_frac'] = float(len(used_b) / nvp)
            del Ab, lb, used_b
        except Exception:
            f['n_boundary_loops'] = 0.0; f['boundary_vert_frac'] = 0.0
        # ---- §4.5.3 non-manifold вершины: степень >2 в графе граничных рёбер ----
        # у нормальной граничной петли каждая вершина имеет ровно 2 инцидентных
        # граничных ребра; степень >2 -> вершина, где встречаются несколько "дыр"
        # или веток границы (типичная примета artifacts/open после плохого шва).
        # Это дополняет уже имеющиеся nonmanifold_edge_ratio/watertight на уровне
        # вершин, а не рёбер, и считается одним bincount без дополнительного графа.
        deg_b = np.bincount(be.ravel(), minlength=nvp)
        f['nonmanifold_vert_ratio'] = float((deg_b > 2).sum() / max(nvp, 1))
        del be, st, deg_b
    else:
        f['n_boundary_loops'] = 0.0; f['boundary_vert_frac'] = 0.0

    two = np.where(counts == 2)[0]
    if len(two):
        st = np.searchsorted(grp, two)
        fa, fb = fids[st], fids[st + 1]
        cosang = np.clip((nrm[fa] * nrm[fb]).sum(1), -1, 1)
        ang = np.degrees(np.arccos(cosang))
        angf = np.degrees(np.arccos(np.abs(cosang)))
        f.update(_stats(ang, 'dihed')); f.update(_stats(angf, 'dihedf'))
        for t in (10, 30, 60, 90):
            f[f'dihed_gt{t}'] = float((ang > t).mean())
        for t in (5, 15, 30, 60):
            f[f'dihedf_gt{t}'] = float((angf > t).mean())
        f['flip_ratio'] = float((cosang < 0).mean())
        hist = np.bincount((angf / 5).astype(np.int32).clip(0, 17), minlength=18).astype(np.float64)
        p = hist / max(hist.sum(), 1)
        f['dihed_entropy'] = float(-(p[p > 0] * np.log(p[p > 0])).sum())
        f['smoothness'] = float(angf.mean() * math.sqrt(nfw) / 100.0)
        # взвешенный по длине ребра угол: устойчивее к мелким треугольникам
        el = np.linalg.norm(Vp[Es[st][:, 0]] - Vp[Es[st][:, 1]], axis=1)
        f['dihedf_areaw'] = float((angf * el).sum() / (el.sum() + 1e-12))
        del fa, fb, cosang, ang, angf, el
    else:
        f.update(_stats([], 'dihed')); f.update(_stats([], 'dihedf'))
        for t in (10, 30, 60, 90):
            f[f'dihed_gt{t}'] = 0.0
        for t in (5, 15, 30, 60):
            f[f'dihedf_gt{t}'] = 0.0
        f.update({'flip_ratio': 0.0, 'dihed_entropy': 0.0, 'smoothness': 0.0, 'dihedf_areaw': 0.0})

    # ---- валентность вершин и угловой дефект (гауссова кривизна) ----
    val = np.bincount(Fi.ravel(), minlength=nvp).astype(np.float32)
    f.update(_stats(val, 'valence'))
    f['valence_le3'] = float((val <= 3).mean())
    f['valence_ge8'] = float((val >= 8).mean())
    e01 = P[:, 1] - P[:, 0]; e12 = P[:, 2] - P[:, 1]; e20 = P[:, 0] - P[:, 2]
    def _ang(u, v):
        cu = (u * v).sum(1) / (np.linalg.norm(u, axis=1) * np.linalg.norm(v, axis=1) + 1e-12)
        return np.arccos(np.clip(cu, -1, 1))
    a0 = _ang(e01, -e20); a1 = _ang(e12, -e01); a2 = _ang(e20, -e12)
    defect = np.full(nvp, 2 * np.pi, dtype=np.float64)
    np.subtract.at(defect, Fi[:, 0], a0)
    np.subtract.at(defect, Fi[:, 1], a1)
    np.subtract.at(defect, Fi[:, 2], a2)
    interior = val > 0
    f.update(_stats(np.abs(defect[interior]), 'defect'))
    f['defect_sum'] = float(defect[interior].sum() / (2 * np.pi))     # ~ эйлерова характеристика
    f['defect_gt05'] = float((np.abs(defect[interior]) > 0.5).mean())
    del defect, val, a0, a1, a2, e01, e12, e20

    del E, Es, fids, grp, counts, two, nrm, P, Vp, Fi, L, aspect, area, area2
    return f

### 5.1. Дешёвые фичи по рендерам (силуэт кадра)

`scale` и `partial` — это буквально «сколько кадра занимает объект» и «на скольких ракурсах
его не видно». Такие фичи считаются из картинки за миллисекунды и дают GBM то,
что CNN приходится выучивать с нуля.

In [ ]:
def image_features(png_path, tile):
    """Силуэтные признаки кадра. Зависят от tile (мозаика режется по нему),
    поэтому кэшируются отдельно от геометрии."""
    f = {'img_ok': 0.0}
    try:
        img = Image.open(png_path).convert('L')
    except Exception:
        return f
    a = np.asarray(img, dtype=np.float32) / 255.0
    t = tile
    f['img_ok'] = 1.0
    covs, comps, edens, stds, bws, bhs = [], [], [], [], [], []
    for k in range(6):
        v = a[(k // 3) * t:(k // 3 + 1) * t, (k % 3) * t:(k % 3 + 1) * t]
        bg = np.median(np.concatenate([v[0], v[-1], v[:, 0], v[:, -1]]))
        mask = np.abs(v - bg) > 0.06
        cov = float(mask.mean()); covs.append(cov)
        if mask.any():
            ys, xs = np.where(mask)
            bws.append((xs.max() - xs.min() + 1) / t)
            bhs.append((ys.max() - ys.min() + 1) / t)
            lab, ncc = ndimage.label(mask)
            if ncc:
                sz = np.bincount(lab.ravel())[1:]
                comps.append(float((sz > 0.01 * sz.sum()).sum()))
            else:
                comps.append(0.0)
        else:
            bws.append(0.0); bhs.append(0.0); comps.append(0.0)
        edens.append(float(np.abs(np.diff(v, axis=1)).mean() + np.abs(np.diff(v, axis=0)).mean()))
        stds.append(float(v.std()))

    def agg(vals, pref):
        v = np.asarray(vals, dtype=np.float64)
        return {f'{pref}_mean': v.mean(), f'{pref}_min': v.min(), f'{pref}_max': v.max(),
                f'{pref}_std': v.std(), f'{pref}_range': v.max() - v.min()}

    f.update(agg(covs, 'cov')); f.update(agg(comps, 'ncc')); f.update(agg(edens, 'edge'))
    f.update(agg(stds, 'pxstd')); f.update(agg(bws, 'bw')); f.update(agg(bhs, 'bh'))
    f['cov_empty_views'] = float((np.asarray(covs) < 0.01).sum())
    f['bbox_fill_view'] = float(np.mean(np.asarray(covs) /
                                        (np.asarray(bws) * np.asarray(bhs) + 1e-6)))
    return f


# =====================================================================================
#  Раздельные кэши.
#
#  В прошлой версии геометрия и силуэты считались одним проходом и кэшировались под
#  именем с `tile`. Из-за этого второй прогон с другим разрешением пересчитывал бы
#  ГЕОМЕТРИЮ — а это самая дорогая часть (сварка вершин, топология, спектр лапласиана,
#  самопересечения на 9633 мешах). Между тем геометрия от разрешения рендера не зависит
#  вообще. Здесь она считается ОДИН раз и переиспользуется обоими прогонами; по `tile`
#  кэшируются только дешёвые силуэтные признаки.
# =====================================================================================

def _mesh_one(args):
    iid, npz = args
    d = {'item_id': iid}
    try:
        d.update(mesh_features(npz))
    except Exception:
        d['geo_ok'] = 0.0
    return d


def _img_one(args):
    iid, png, tile = args
    d = {'item_id': iid}
    try:
        d.update(image_features(png, tile))
    except Exception:
        d['img_ok'] = 0.0
    return d


def _chunked_build(jobs, worker, final, ckpt, desc):
    """Общий каркас: чанками, с чекпоинтом, с резюмированием после обрыва."""
    if final.exists():
        df = pd.read_parquet(final)
        R.ok(f'{desc}: из кэша {df.shape}')
        return df
    done = pd.read_parquet(ckpt) if ckpt.exists() else pd.DataFrame(columns=['item_id'])
    have = set(done['item_id'].astype(str)) if len(done) else set()
    todo = [j for j in jobs if j[0] not in have]
    R.kv(**{desc: f'готово {len(have)}, осталось {len(todo)}, воркеров {N_WORKERS}'})
    rows = [done] if len(done) else []
    for s in range(0, len(todo), CHUNK):
        part = todo[s:s + CHUNK]
        try:
            ctx = mp.get_context('fork')
        except ValueError:
            ctx = None
        try:
            with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=ctx) as ex:
                got = list(tqdm(ex.map(worker, part, chunksize=4), total=len(part),
                                desc=f'{desc} {s + len(part)}/{len(todo)}', leave=False))
        except (BrokenProcessPool, OSError, MemoryError) as e:
            R.warn(f'пул упал ({type(e).__name__}), чанк считается последовательно')
            got = [worker(j) for j in tqdm(part, desc='serial', leave=False)]
        rows.append(pd.DataFrame(got))
        _atomic(ckpt, lambda p: pd.concat(rows, ignore_index=True).to_parquet(p, index=False))
        gc.collect()
    df = pd.concat(rows, ignore_index=True).drop_duplicates('item_id') if rows else \
        pd.DataFrame({'item_id': [j[0] for j in jobs]})
    _atomic(final, lambda p: df.to_parquet(p, index=False))
    if ckpt.exists():
        ckpt.unlink()
    return df


def build_mesh_features(ids, npz_dir, split):
    """Кэш НЕ содержит tile: геометрия одна на все прогоны."""
    jobs = [(str(i), str(Path(npz_dir) / f'{i}.npz')) for i in ids]
    df = _chunked_build(jobs, _mesh_one,
                        CFG.cache / f'mesh_{split}_{FEAT_VER}.parquet',
                        CFG.cache / f'mesh_{split}_{FEAT_VER}_partial.parquet',
                        f'геометрия {split}')
    return df.set_index('item_id').reindex([str(i) for i in ids]).reset_index().fillna(0.0)


def build_image_features(ids, img_dir, split, tile):
    jobs = [(str(i), str(Path(img_dir) / f'{i}.png'), tile) for i in ids]
    df = _chunked_build(jobs, _img_one,
                        CFG.cache / f'imgf_{split}_{tile}.parquet',
                        CFG.cache / f'imgf_{split}_{tile}_partial.parquet',
                        f'силуэты {split} @{tile}')
    return df.set_index('item_id').reindex([str(i) for i in ids]).reset_index().fillna(0.0)


R.section('Геометрические признаки', f'версия {FEAT_VER}, считаются один раз на оба прогона')
mesh_tr = build_mesh_features(train_df['item_id'].tolist(), TRAIN_DIR, 'train')
mesh_te = build_mesh_features(test_df['item_id'].tolist(), TEST_DIR, 'test')
R.kv(**{'геометрия train': mesh_tr.shape, 'геометрия test': mesh_te.shape})


### 5.2. CLIP zero-shot как признак (прежде всего для `abstract`)

`abstract` описан в условии буквально текстом: «3D-надписи, графики, таблицы, схемы, диаграммы,
поддержки для 3D-печати, объекты в стиле Minecraft». Это не геометрическое свойство, а
семантическая категория — её невозможно вывести из двугранных углов, и именно поэтому
baseline на облаке точек её терял. CLIP обучен сопоставлять изображение и текст, поэтому
нужные категории задаются **прямо промптами**, без единой размеченной картинки.

Каждый из 6 видов кодируем CLIP-энкодером, берём косинусную близость к ~30 текстовым
описаниям, агрегируем по видам (среднее / максимум / разброс) и добавляем как обычные
признаки в GBM и стэкинг. Отдельно считаем **контрастные** признаки — разность
«абстрактные» минус «предметные» промпты: сигнал несут именно они, а не абсолютные косинусы
(у CLIP они лежат в узком диапазоне и сами по себе почти неинформативны).

Стоимость: один прогон ViT-B/32 по 58 тысячам видов — единицы минут, результат кэшируется.
Риска переобучения нет: веса CLIP не трогаем, метки не используем.

In [ ]:
CLIP_PROMPTS = {
    'abs_text':     ['3D text lettering', 'a sign with written words', 'an extruded logo'],
    'abs_chart':    ['a bar chart', 'a pie chart', 'a graph plot', 'a diagram', 'a flowchart',
                     'a table with data', 'an infographic'],
    'abs_support':  ['3D printing support structure', 'scaffolding lattice'],
    'abs_voxel':    ['minecraft voxel blocks', 'blocky pixelated cubes'],
    'obj_single':   ['a 3D model of a single object', 'a product render on white background'],
    'obj_semantic': ['a chair', 'a car', 'a building', 'a plant', 'a character figure',
                     'a tool', 'furniture'],
    'p_lowpoly':    ['a low-poly 3D model with visible flat facets'],
    'p_noisy':      ['a noisy 3D scan with rough surface', 'a point cloud scan'],
    'p_broken':     ['a broken mesh with holes and artifacts'],
    'p_hollow':     ['a hollow thin shell', 'a flat plane'],
    'p_multi':      ['several separate objects scattered apart', 'a collection of many objects'],
    'p_small':      ['a tiny object in the middle of a large empty frame'],
    'p_smooth':     ['a smooth clean 3D render of one object'],
}


class _ViewsDataset(torch.utils.data.Dataset):
    """Читает ИСХОДНЫЙ рендер и режет на 6 квадратов нужного размера.

    Источник — оригинальный PNG, а не кэш под конкретный tile, поэтому CLIP-признаки
    не зависят от конфигурации прогона и считаются один раз на оба.
    """
    def __init__(self, ids, src_dir, size):
        self.ids, self.dir, self.size = [str(i) for i in ids], Path(src_dir), size

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        p = self.dir / f'{self.ids[i]}.png'
        s = self.size
        if p.exists():
            vs = [np.asarray(v.resize((s, s), Image.BILINEAR), dtype=np.uint8)
                  for v in split_views(Image.open(p).convert('RGB'))]
        else:
            vs = [np.full((s, s, 3), 255, np.uint8)] * 6
        return torch.from_numpy(np.stack(vs)), self.ids[i]


def compute_clip_features(ids, src_dir, split):
    """Узкое место прошлого прогона: декодирование PNG шло в один поток в теле цикла,
    и 8964 объекта заняли 83 минуты при простаивающем GPU. Здесь чтение вынесено
    в DataLoader с num_workers — то же самое считается в разы быстрее."""
    import open_clip
    name, pretrained = CFG.clip_model
    model, _, _ = open_clip.create_model_and_transforms(name, pretrained=pretrained)
    tokenizer = open_clip.get_tokenizer(name)
    model = model.to(CFG.device).eval()

    groups = list(CLIP_PROMPTS)
    flat, owner = [], []
    for g in groups:
        flat += CLIP_PROMPTS[g]; owner += [g] * len(CLIP_PROMPTS[g])
    with torch.no_grad():
        tf = model.encode_text(tokenizer(flat).to(CFG.device)).float()
        tf = tf / tf.norm(dim=-1, keepdim=True)
    owner = np.array(owner)

    size = model.visual.image_size
    size = size[0] if isinstance(size, (tuple, list)) else int(size)
    mean = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=CFG.device).view(1, 3, 1, 1)
    std = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=CFG.device).view(1, 3, 1, 1)

    ds = _ViewsDataset(ids, src_dir, size)
    dl = torch.utils.data.DataLoader(ds, batch_size=32, shuffle=False,
                                     num_workers=CFG.num_workers, pin_memory=True)
    rows = []
    for x, iids in tqdm(dl, desc=f'CLIP {split}'):
        x = x.to(CFG.device, non_blocking=True)
        B = x.shape[0]
        x = x.permute(0, 1, 4, 2, 3).reshape(B * 6, 3, size, size).float() / 255.0
        x = (x - mean) / std
        with torch.no_grad():
            im = model.encode_image(x).float()
        im = im / im.norm(dim=-1, keepdim=True)
        sim = (im @ tf.T).view(B, 6, -1).cpu().numpy()
        gsim = np.stack([sim[:, :, owner == g].max(-1) for g in groups], -1)
        feat = {}
        for j, g in enumerate(groups):
            v = gsim[:, :, j]
            feat[f'clip_{g}_mean'] = v.mean(1)
            feat[f'clip_{g}_max'] = v.max(1)
            feat[f'clip_{g}_std'] = v.std(1)
        abs_max = np.stack([feat[f'clip_{g}_max'] for g in groups if g.startswith('abs_')], 1).max(1)
        obj_max = np.stack([feat[f'clip_{g}_max'] for g in groups if g.startswith('obj_')], 1).max(1)
        def_max = np.stack([feat[f'clip_{g}_max'] for g in groups
                            if g.startswith('p_') and g != 'p_smooth'], 1).max(1)
        feat['clip_abstract_margin'] = abs_max - obj_max
        feat['clip_abstract_prob'] = 1 / (1 + np.exp(-100 * (abs_max - obj_max)))
        feat['clip_defect_margin'] = def_max - feat['clip_p_smooth_max']
        rows.append(pd.DataFrame(feat, index=list(iids)))
    del model
    gc.collect(); torch.cuda.empty_cache()
    return pd.concat(rows).rename_axis('item_id').reset_index()


R.section('CLIP zero-shot', 'считается один раз, от разрешения прогона не зависит')
if CFG.use_clip:
    clip_tr = cached('clip_train_orig',
                     lambda: compute_clip_features(train_df['item_id'].tolist(), TRAIN_DIR, 'train'))
    clip_te = cached('clip_test_orig',
                     lambda: compute_clip_features(test_df['item_id'].tolist(), TEST_DIR, 'test'))
    R.kv(**{'CLIP-признаков': clip_tr.shape[1] - 1})
    _m = train_df.merge(clip_tr, on='item_id', how='left').fillna(0)
    cc = pd.Series({c: np.corrcoef(_m[c], _m['abstract'])[0, 1]
                    for c in clip_tr.columns if c != 'item_id'}).sort_values(key=abs, ascending=False)
    print('  корреляция с abstract (топ-5):')
    print('  ' + cc.head(5).round(3).to_string().replace('\n', '\n  '))
else:
    clip_tr = clip_te = None
    R.warn('CLIP выключен')


# =============================== сборка таблицы признаков ===============================
FEAT_COLS = None          # фиксируется на первом прогоне и переиспользуется вторым


def assemble_features(img_train, img_test, tile):
    """Геометрия (одна на все прогоны) + силуэты под этот tile + CLIP."""
    global FEAT_COLS
    ftr = mesh_tr.merge(build_image_features(train_df['item_id'].tolist(), img_train, 'train', tile),
                        on='item_id', how='left')
    fte = mesh_te.merge(build_image_features(test_df['item_id'].tolist(), img_test, 'test', tile),
                        on='item_id', how='left')
    if clip_tr is not None:
        ftr = ftr.merge(clip_tr, on='item_id', how='left')
        fte = fte.merge(clip_te, on='item_id', how='left')

    cols = [c for c in ftr.columns if c != 'item_id' and c in fte.columns]
    ftr[cols] = ftr[cols].replace([np.inf, -np.inf], 0).fillna(0).astype(np.float32)
    fte[cols] = fte[cols].replace([np.inf, -np.inf], 0).fillna(0).astype(np.float32)

    if FEAT_COLS is None:
        # Список фич фиксируется ОДИН раз. Иначе у второго прогона размерность входа
        # гео-ветки окажется другой, и общий стэкинг поверх обоих развалится.
        import hashlib
        keep, seen, const, dup = [], {}, [], []
        for c in cols:
            v = np.nan_to_num(ftr[c].values.astype(np.float64))
            if v.std() == 0:
                const.append(c); continue
            key = hashlib.md5(np.ascontiguousarray(np.round(v, 9)).tobytes()).hexdigest()
            if key in seen:
                dup.append(c); continue
            seen[key] = c; keep.append(c)
        FEAT_COLS = cached(f'feat_cols_{FEAT_VER}', lambda: keep)
        R.kv(**{'признаков всего': len(cols), 'после дедупликации': len(FEAT_COLS),
                'спектральных': sum(c.startswith('spec') for c in FEAT_COLS),
                'CLIP': sum(c.startswith('clip') for c in FEAT_COLS)})
    for df_ in (ftr, fte):
        for c in FEAT_COLS:
            if c not in df_.columns:
                df_[c] = 0.0
    return ftr, fte


In [ ]:
# Дедупликация и фиксация FEAT_COLS происходят внутри assemble_features (§5.2):
# список признаков обязан быть один на оба прогона, иначе у второго окажется
# другая размерность входа гео-ветки.
R.ok('список признаков фиксируется на первом прогоне (§9.1)')


## 6. Фолды: MultilabelStratifiedKFold

Обычный `train_test_split(stratify=quality)` из baseline не сохраняет доли редких дефектов
(`intersection`, `scale` — по 107 объектов). Итеративная мультилейбл-стратификация держит
долю **каждой** из 11 меток в каждом фолде — иначе OOF-пороги переобучаются на шум.

In [ ]:
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

mskf = MultilabelStratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
train_df['fold'] = -1
for k, (_, vi) in enumerate(mskf.split(train_df, train_df[CFG.target_cols].values)):
    train_df.loc[train_df.index[vi], 'fold'] = k
display(train_df.groupby('fold')[CFG.target_cols].mean().round(4))

## 7. Датасет и аугментации

**Осознанный отказ от вращений.** Метки `partial`/`open`/`set` определены относительно
конкретных 6 ракурсов («не видно с некоторых ракурсов»), поэтому в baseline случайная
ротация облака точек уничтожала сигнал. Здесь порядок видов канонический, а аугментации
ограничены теми, что не меняют смысла: флип каждого вида, лёгкий сдвиг/масштаб, jitter яркости,
и `view-dropout` (зануление одного вида) — он же регуляризатор для `partial`.

In [ ]:
from torch.utils.data import Dataset, DataLoader

MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def load_views(path, tile):
    a = np.asarray(Image.open(path).convert('RGB'), dtype=np.uint8)
    t = tile
    return np.stack([a[(k // 3) * t:(k // 3 + 1) * t, (k % 3) * t:(k % 3 + 1) * t]
                     for k in range(6)])


def symmetry(v, rot=0, mirror=False, flip_ud=False):
    """Точная симметрия съёмочного стенда: 4 азимута с шагом 90° + верх + низ.

    Множество из шести кадров не меняется — меняются только их порядок и ориентация,
    поэтому ВСЕ 11 меток инвариантны. Это 4*2*2 = 16 преобразований без риска испортить
    разметку, в отличие от зума (ломает `scale`), пиксельного шума (ломает `noisy`,
    вес 0.274) и обнуления вида (создаёт условие `partial` при метке 0).

    Прежний флип `v[:, :, ::-1]` отражал кадры, но НЕ разворачивал порядок азимутов —
    получалась конфигурация, которой не соответствует ни один реальный объект.
    """
    az, po = list(CFG.azimuth_idx), list(CFG.pole_idx)
    out = [v[az[(k + rot) % 4]] for k in range(4)]
    top, bot = po
    pt = np.rot90(v[top], -rot, axes=(0, 1))
    pb = np.rot90(v[bot],  rot, axes=(0, 1))
    if mirror:
        out = [out[0]] + out[1:][::-1]
        out = [x[:, ::-1] for x in out]
        pt, pb = pt[:, ::-1], pb[:, ::-1]
    if flip_ud:
        out = [x[::-1] for x in out]
        pt, pb = pb[::-1], pt[::-1]
    return np.ascontiguousarray(np.stack(out + [pt, pb]))


SYM_ALL = [(r, m, u) for r in range(4) for m in (False, True) for u in (False, True)]
# В TTA переворот верх-низ не берём: тест снят в одной ориентации, и подмешивать
# перевёрнутые кадры на инференсе — это сдвиг распределения, а не усреднение шума.
CFG.tta_syms = [(r, m, False) for r in range(4) for m in (False, True)]


class MeshViewDataset(Dataset):
    def __init__(self, df, img_dir, feats, train=True, labels=True):
        self.ids = df['item_id'].tolist()
        self.dir = Path(img_dir)
        self.train = train
        self.labels = df[CFG.target_cols].values.astype(np.float32) if labels else None
        self.feats = feats.set_index('item_id').reindex(self.ids)[FEAT_COLS] \
                          .fillna(0.0).values.astype(np.float32)

    def __len__(self):
        return len(self.ids)

    def _aug(self, v):
        rng = np.random
        if CFG.sym_aug:
            v = symmetry(v, *SYM_ALL[rng.randint(len(SYM_ALL))])
        elif rng.rand() < 0.5:
            v = symmetry(v, 0, True, False)
        if rng.rand() < 0.3:
            # сдвиг с заливкой фоном, а не np.roll: заворачивание краёв вклеивает
            # кусок объекта с противоположной стороны кадра
            ys, xs = (int(z) for z in rng.randint(-8, 9, size=2))
            out = np.full_like(v, 255)
            H, W = v.shape[1], v.shape[2]
            y0, y1 = max(0, ys), min(H, H + ys)
            x0, x1 = max(0, xs), min(W, W + xs)
            out[:, y0:y1, x0:x1] = v[:, y0 - ys:y1 - ys, x0 - xs:x1 - xs]
            v = out
        if CFG.view_dropout and rng.rand() < CFG.view_dropout:
            v = v.copy(); v[rng.randint(6)] = 255
        v = v.astype(np.float32)
        if rng.rand() < 0.3:
            v = np.clip(v * rng.uniform(0.85, 1.15) + rng.uniform(-15, 15), 0, 255)
        if CFG.pixel_noise and rng.rand() < CFG.pixel_noise:
            v = np.clip(v + rng.normal(0, 4, v.shape), 0, 255)
        return v

    def __getitem__(self, i):
        p = self.dir / f'{self.ids[i]}.png'
        v = load_views(p, CFG.tile) if p.exists() else \
            np.full((6, CFG.tile, CFG.tile, 3), 255, np.uint8)
        v = self._aug(v) if self.train else v.astype(np.float32)
        v = ((v / 255.0 - MEAN) / STD).astype(np.float32)
        out = {'views': torch.from_numpy(np.ascontiguousarray(v)).permute(0, 3, 1, 2),
               'feats': torch.from_numpy(self.feats[i]), 'item_id': self.ids[i]}
        if self.labels is not None:
            out['y'] = torch.from_numpy(self.labels[i])
        return out


def _sym_torch(v, rot, mirror, flip_ud=False):
    az, po = list(CFG.azimuth_idx), list(CFG.pole_idx)
    v = v[:, [az[(k + rot) % 4] for k in range(4)] + po]
    if rot:
        v = torch.cat([v[:, :4], torch.rot90(v[:, 4:5], -rot, dims=(-2, -1)),
                       torch.rot90(v[:, 5:6], rot, dims=(-2, -1))], 1)
    if mirror:
        v = torch.cat([v[:, :1], v[:, 1:4].flip(1), v[:, 4:]], 1).flip(-1)
    if flip_ud:
        v = torch.cat([v[:, :4], v[:, 5:6], v[:, 4:5]], 1).flip(-2)
    return v


@torch.no_grad()
def predict(model, loader, tta=None, desc='predict'):
    """tta=1 — один проход (мониторинг эпох), иначе TTA по симметриям стенда."""
    model.eval()
    syms = [(0, False, False)] if tta == 1 else CFG.tta_syms
    probs, ids = [], []
    for b in tqdm(loader, desc=f'{desc} (TTA x{len(syms)})', leave=False):
        v = b['views'].to(CFG.device, non_blocking=True)
        f = b['feats'].to(CFG.device, non_blocking=True)
        acc = 0
        with torch.cuda.amp.autocast(enabled=CFG.amp):
            for s in syms:
                acc = acc + torch.sigmoid(model(_sym_torch(v, *s), f)).float()
        probs.append((acc / len(syms)).cpu().numpy()); ids.extend(b['item_id'])
    return np.vstack(probs), ids


def check_view_layout(n=200):
    """Симметрии осмысленны, только если виды 0-3 — азимуты, 4-5 — полюса.
    Проверяем данными: соседние азимуты должны быть похожи сильнее, чем азимут и полюс."""
    acc, cnt = np.zeros((6, 6)), 0
    for iid in train_df['item_id'].head(n):
        p = Path(IMG_TRAIN) / f'{iid}.png'
        if not p.exists():
            continue
        x = load_views(p, CFG.tile).astype(np.float32).mean(-1)
        m = (np.abs(x - np.median(x)) > 15).reshape(6, -1).astype(np.float32)
        m -= m.mean(1, keepdims=True)
        nrm = np.linalg.norm(m, axis=1) + 1e-9
        acc += (m @ m.T) / np.outer(nrm, nrm); cnt += 1
    S = acc / max(cnt, 1)
    ring = np.mean([S[i, (i + 1) % 4] for i in range(4)])
    cross = np.mean([S[i, j] for i in CFG.azimuth_idx for j in CFG.pole_idx])
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    im = ax[0].imshow(S, cmap='viridis'); ax[0].grid(False)
    ax[0].set_title('Сходство силуэтов между видами')
    for i in range(6):
        for j in range(6):
            ax[0].text(j, i, f'{S[i, j]:.2f}', ha='center', va='center', fontsize=7,
                       color='w' if S[i, j] < S.max() * .6 else 'k')
    plt.colorbar(im, ax=ax[0], fraction=.046)
    ax[1].bar(['соседние\nазимуты', 'азимут-полюс'], [ring, cross],
              color=[PAL['good'], PAL['bad']])
    ax[1].set_title('Кольцевая структура')
    plt.tight_layout(); plt.show()
    if ring > cross + 0.02:
        R.ok(f'раскладка подтверждена (кольцо {ring:.3f} > крест {cross:.3f}) — '
             f'{len(SYM_ALL)} симметрий включены, TTA x{len(CFG.tta_syms)}')
    else:
        CFG.sym_aug = False
        CFG.tta_syms = [(0, False, False), (0, True, False)]
        R.warn('кольцевая структура не подтверждена — симметрии выключены')
    return S


_t = np.random.randint(0, 255, (6, 64, 64, 3), dtype=np.uint8)
assert np.array_equal(symmetry(_t, 0, False, False), _t)
assert np.array_equal(symmetry(symmetry(_t, 1), 3), _t)
assert np.array_equal(symmetry(symmetry(_t, 0, True), 0, True), _t)
assert np.array_equal(symmetry(symmetry(_t, 0, False, True), 0, False, True), _t)
assert len({tuple(sorted(int(x.sum()) for x in symmetry(_t, *s))) for s in SYM_ALL}) == 1
R.ok(f'{len(SYM_ALL)} симметрий: множество кадров сохраняется, преобразования обратимы')


## 8. Модель

```
  6 рендеров ─┬─ grid: одна мозаика 2×3 ─┐
              └─ multiview: shared backbone ×6 ─ attention pooling ─┤
                                                                    ├─ concat ─ MLP ─ 11 логитов
  ~120 гео/визуальных фич ─ BatchNorm ─ MLP(128) ─────────────────────┘
```

* **Shared backbone + attention pooling** (MVCNN, Su et al. ICCV'15 + gated attention MIL,
  Ilse et al. ICML'18): дефект часто виден только с одного ракурса, max/attention-пулинг это ловит,
  в отличие от усреднения.
* **Гео-ветка внутри сети**, а не только в GBM: `boundary_edge_ratio` даёт `open` почти даром,
  и CNN не тратит ёмкость на его выучивание.
* **Отдельная голова `quality`** — 11-й выход. Она может «не согласиться» с OR-правилом,
  и на OOF мы выберем, что выгоднее (см. §11).
* **Loss = 0.5·ASL + 0.5·BCE(pos_weight)**: Asymmetric Loss (Ridnik et al. ICCV'21) — SOTA для
  мультилейбла с дисбалансом негативов; BCE с `pos_weight` не даёт редким классам схлопнуться в 0.

In [ ]:
import timm


class MLDecoder(nn.Module):
    """Голова классификации с запросом на класс (Ridnik et al., WACV 2023, arXiv 2111.12933).

    Зачем именно здесь. До сих пор каждый вид сжимался глобальным пулингом в один
    вектор, и только потом решалось, какие метки поставить. Для `noisy` или `simple`
    это нормально — они про объект целиком. Но `open` (F1 0.30), `artifacts` (0.33)
    и `intersection` (0.15) — дефекты **локальные**: дыра занимает малую долю кадра,
    и при усреднении по всем патчам её сигнал разбавляется в сотни раз.

    ML-Decoder заводит обучаемый запрос на каждый класс и даёт ему через cross-attention
    самому выбрать, на какие участки каких видов смотреть. Запрос `open` может
    сосредоточиться на краях силуэта, запрос `set` — на разнесённых кусках. Self-attention
    между запросами в оригинальной статье убран как избыточный, поэтому стоимость линейна
    по числу токенов: 11 запросов почти ничего не стоят.

    Токены: патч-сетка каждого вида ужимается адаптивным пулингом до
    `mld_tokens` x `mld_tokens`, к ней прибавляется кодирование номера вида, и все шесть
    видов склеиваются в одну последовательность. При 8x8 это 384 токена независимо от
    разрешения — то есть 448 не удорожает голову, только бэкбон.
    """

    def __init__(self, d_in, n_out, dim=384, layers=2, heads=8):
        super().__init__()
        self.proj = nn.Linear(d_in, dim) if d_in != dim else nn.Identity()
        self.query = nn.Parameter(torch.zeros(1, n_out, dim))
        nn.init.trunc_normal_(self.query, std=0.02)
        self.view_emb = nn.Parameter(torch.zeros(1, 6, 1, dim))
        nn.init.trunc_normal_(self.view_emb, std=0.02)
        self.blocks = nn.ModuleList()
        for _ in range(layers):
            self.blocks.append(nn.ModuleDict({
                'norm_q': nn.LayerNorm(dim), 'norm_k': nn.LayerNorm(dim),
                'attn': nn.MultiheadAttention(dim, heads, dropout=0.1, batch_first=True),
                'norm_f': nn.LayerNorm(dim),
                'ffn': nn.Sequential(nn.Linear(dim, 2 * dim), nn.GELU(),
                                     nn.Dropout(0.1), nn.Linear(2 * dim, dim))}))
        self.norm_out = nn.LayerNorm(dim)

    def forward(self, tokens, extra=None):
        """tokens: (B, 6, T, d_in) — патч-токены каждого вида.
        extra:  (B, d)  — гео-вектор, добавляется как ещё один ключ."""
        B, V, T, _ = tokens.shape
        x = self.proj(tokens) + self.view_emb[:, :V]
        x = x.reshape(B, V * T, -1)
        if extra is not None:
            x = torch.cat([x, extra.unsqueeze(1)], 1)
        q = self.query.expand(B, -1, -1)
        att_last = None
        for blk in self.blocks:
            a, w = blk['attn'](blk['norm_q'](q), blk['norm_k'](x), blk['norm_k'](x),
                               need_weights=True, average_attn_weights=True)
            q = q + a
            q = q + blk['ffn'](blk['norm_f'](q))
            att_last = w
        return self.norm_out(q), att_last          # (B, n_out, dim), (B, n_out, L)


class MultiViewNet(nn.Module):
    def __init__(self, n_feats, n_out=CFG.n_targets):
        super().__init__()
        kw = dict(pretrained=True, num_classes=0, drop_rate=CFG.drop_rate)
        if IS_VIT:
            kw['img_size'] = CFG.tile
        self.backbone = timm.create_model(CFG.backbone, **kw)
        d = self.backbone.num_features
        self.d = d
        self.geo = nn.Sequential(
            nn.BatchNorm1d(n_feats), nn.Linear(n_feats, 256), nn.SiLU(), nn.Dropout(0.2),
            nn.Linear(256, 128), nn.SiLU())

        if CFG.use_mldecoder:
            self.dec = MLDecoder(d, n_out, dim=CFG.mld_dim, layers=CFG.mld_layers)
            self.geo_to_tok = nn.Linear(128, CFG.mld_dim)
            # голова: по одному линейному выходу на класс, применённому к своему запросу
            self.cls_w = nn.Parameter(torch.zeros(n_out, CFG.mld_dim))
            self.cls_b = nn.Parameter(torch.zeros(n_out))
            nn.init.trunc_normal_(self.cls_w, std=0.02)
        else:
            self.att_v = nn.Sequential(nn.Linear(d, 256), nn.Tanh())
            self.att_u = nn.Sequential(nn.Linear(d, 256), nn.Sigmoid())
            self.att_w = nn.Linear(256, 1)
            self.head = nn.Sequential(nn.Linear(2 * d + 128, 512), nn.SiLU(),
                                      nn.Dropout(0.3), nn.Linear(512, n_out))

    def _patch_tokens(self, x):
        """(B*V, 3, H, W) -> (B*V, T, d): патч-токены, ужатые до mld_tokens^2."""
        n = CFG.mld_tokens
        if IS_VIT:
            f = self.backbone.forward_features(x)          # (N, 1+r+P, d)
            npre = f.shape[1] - int(math.isqrt(f.shape[1])) ** 2
            g = f[:, npre:]
            s = int(math.isqrt(g.shape[1]))
            g = g[:, :s * s].transpose(1, 2).reshape(g.shape[0], -1, s, s)
        else:
            g = self.backbone.forward_features(x)          # (N, d, H', W')
        g = F.adaptive_avg_pool2d(g, (n, n))
        return g.flatten(2).transpose(1, 2)                # (N, n*n, d)

    def embed(self, views, feats):
        B, V = views.shape[:2]
        flat = views.flatten(0, 1)
        gv = self.geo(feats)
        if CFG.use_mldecoder:
            tok = self._patch_tokens(flat).view(B, V, -1, self.d)
            q, att = self.dec(tok, self.geo_to_tok(gv))
            logits = (q * self.cls_w.unsqueeze(0)).sum(-1) + self.cls_b
            return logits, att
        z = self.backbone(flat).view(B, V, -1)
        a = self.att_w(self.att_v(z) * self.att_u(z))
        w = torch.softmax(a, 1)
        pooled = torch.cat([(z * w).sum(1), z.max(1).values], -1)
        return self.head(torch.cat([pooled, gv], 1)), w.squeeze(-1)

    def forward(self, views, feats, return_att=False):
        logits, att = self.embed(views, feats)
        return (logits, att) if return_att else logits


class AsymmetricLoss(nn.Module):
    """Ridnik et al., ICCV 2021 — как в прогоне, давшем 14.987."""
    def __init__(self, gamma_neg=4.0, gamma_pos=0.0, clip=0.05, eps=1e-8):
        super().__init__()
        self.gn, self.gp, self.clip, self.eps = gamma_neg, gamma_pos, clip, eps

    def forward(self, logits, y):
        p = torch.sigmoid(logits)
        p_neg = (1 - p + self.clip).clamp(max=1.0)
        loss = y * torch.log(p.clamp(min=self.eps)) + (1 - y) * torch.log(p_neg.clamp(min=self.eps))
        pt = p * y + (1 - p) * (1 - y)
        gamma = self.gp * y + self.gn * (1 - y)
        return -(loss * (1 - pt).pow(gamma)).mean()


class ModelEMA:
    def __init__(self, model, decay=0.999):
        self.ema = copy.deepcopy(model).eval()
        for p in self.ema.parameters():
            p.requires_grad_(False)
        self.decay = decay

    @torch.no_grad()
    def update(self, model, step=None):
        d = self.decay if step is None else min(self.decay, (1 + step) / (10 + step))
        msd = model.state_dict()
        for k, v in self.ema.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(d).add_(msd[k].detach(), alpha=1 - d)
            else:
                v.copy_(msd[k])

    def state_dict(self): return self.ema.state_dict()
    def load_state_dict(self, sd): self.ema.load_state_dict(sd)


def _selftest_model():
    m = MultiViewNet(n_feats=16).to(CFG.device).eval()
    v = torch.randn(2, 6, 3, CFG.tile, CFG.tile, device=CFG.device)
    f = torch.randn(2, 16, device=CFG.device)
    with torch.no_grad():
        o, a = m(v, f, return_att=True)
    R.ok(f'модель собрана: logits {tuple(o.shape)}, attention {tuple(a.shape)}, '
         f'параметров {sum(p.numel() for p in m.parameters()) / 1e6:.1f}M')
    if CFG.use_mldecoder:
        R.kv(**{'токенов в декодере': f'6 видов x {CFG.mld_tokens}^2 = '
                                      f'{6 * CFG.mld_tokens ** 2} + 1 гео'})
    del m, v, f; gc.collect(); torch.cuda.empty_cache()


_selftest_model()


## 9. Обучение CNN (OOF + предсказания на тесте)

AdamW + cosine с warmup, AMP, повышенный LR для головы, grad clipping.
На каждой эпохе печатается **соревновательная метрика** на валидации с per-class F1 —
видно, какой класс тянет вниз, а не абстрактный loss.

In [ ]:
from torch.cuda.amp import autocast, GradScaler


def per_class_f1(y, p):
    return pd.Series([f1_score(y[:, i], p[:, i], zero_division=0) for i in range(11)],
                     index=CFG.target_cols)


def exact_f1_threshold(y, p, plateau=0.995):
    """Точный argmax F1 по порогу за один проход сортировки.

    Разрез допустим только там, где значение p МЕНЯЕТСЯ: иначе порог попадает между
    двумя одинаковыми вероятностями и `p > th` выбрасывает оба объекта вместо одного.
    При насыщенных сигмоидах и усреднении по фолдам совпадения массовые.
    """
    y = np.asarray(y).astype(np.int32); p = np.asarray(p, np.float64)
    P = int(y.sum())
    if P == 0:
        return 0.5, 0.0
    o = np.argsort(-p, kind='stable'); ys, ps = y[o], p[o]
    f1 = 2 * np.cumsum(ys) / (np.arange(1, len(y) + 1) + P)
    fb = float(f1.max())
    cut = np.empty(len(ps), bool); cut[-1] = True; cut[:-1] = ps[:-1] > ps[1:]
    ok = np.where((f1 >= plateau * fb) & cut)[0]
    if not len(ok):
        ok = np.where(cut)[0][[int(np.argmax(f1[cut]))]]
    i = int(ok[len(ok) // 2])
    return float((ps[i] + ps[i + 1]) / 2 if i + 1 < len(ps) else ps[i] - 1e-6), fb


def quick_thresholds(y, prob):
    return np.array([exact_f1_threshold(y[:, i], prob[:, i])[0] for i in range(11)])


def _fmt_eta(sec):
    sec = int(max(sec, 0))
    return f'{sec // 3600}ч {sec % 3600 // 60:02d}м' if sec >= 3600 else f'{sec // 60}м {sec % 60:02d}с'


def live_dashboard(hist, fold):
    if not hist:
        return
    h = pd.DataFrame(hist)
    fig, ax = plt.subplots(1, 3, figsize=(16, 4))
    ax[0].plot(h['epoch'], h['val_metric'], '-o', ms=4, color=PAL['main'], label='raw')
    if h['val_metric_ema'].notna().any():
        ax[0].plot(h['epoch'], h['val_metric_ema'], '-s', ms=4, color=PAL['alt'], label='EMA')
    ax[0].axhline(14.25, color=PAL['grey'], ls=':', lw=1, label='CNN в v5')
    ax[0].set_xlabel('эпоха'); ax[0].legend(fontsize=8)
    ax[0].set_title(f'{CFG.run_tag} fold {fold}: метрика')
    ax[1].plot(h['epoch'], 10 * h['f1_quality'], '-o', ms=4, color=PAL['bad'], label='quality')
    ax[1].plot(h['epoch'], 10 * h['f1_artefacts'], '-o', ms=4, color=PAL['warn'], label='artefacts')
    ax[1].set_xlabel('эпоха'); ax[1].legend(fontsize=8); ax[1].set_title('Половины метрики')
    f1c = [c for c in h.columns if c.startswith('f1_') and c not in ('f1_quality', 'f1_artefacts')]
    last = h.iloc[-1][f1c].astype(float).sort_values()
    ax[2].barh([c[3:] for c in last.index], last.values,
               color=[PAL['bad'] if v < .4 else PAL['warn'] if v < .6 else PAL['good']
                      for v in last.values])
    ax[2].set_xlim(0, 1); ax[2].set_title(f'per-class F1, эпоха {int(h["epoch"].iloc[-1])}')
    plt.tight_layout(); plt.show()


def train_fold(fold):
    if CFG.resume and has_artifact(rt(f'fold{fold}_preds')):
        d = load_artifact(rt(f'fold{fold}_preds'))
        R.ok(f'фолд {fold} взят из кэша (метрика {d["ens"]:.3f}) — обучение пропускаем')
        return (d['oof_ids'], d['oof']), (d['te_ids'], d['te'])

    R.section(f'{CFG.run_tag}: фолд {fold} / {CFG.n_folds}',
              f'{CFG.backbone.split(".")[0]} @ {CFG.tile} | эпох {CFG.epochs}')
    seed_everything(CFG.seed + fold)
    tr = train_df[train_df.fold != fold].reset_index(drop=True)
    va = train_df[train_df.fold == fold].reset_index(drop=True)
    ltr = DataLoader(MeshViewDataset(tr, IMG_TRAIN, feat_tr, train=True),
                     batch_size=CFG.batch_size, shuffle=True, drop_last=True,
                     num_workers=CFG.num_workers, pin_memory=True, persistent_workers=True)
    lva = DataLoader(MeshViewDataset(va, IMG_TRAIN, feat_tr, train=False),
                     batch_size=CFG.batch_size * 2, num_workers=CFG.num_workers, pin_memory=True)
    R.kv(**{'объектов train / val': f'{len(tr)} / {len(va)}',
            'шагов на эпоху': f'{len(ltr)} (оптимизаторских {len(ltr) // CFG.accum})'})

    model = MultiViewNet(len(FEAT_COLS)).to(CFG.device)
    ema = ModelEMA(model, CFG.ema_decay)
    head = [p for n_, p in model.named_parameters() if not n_.startswith('backbone')]
    head_ids = {id(p) for p in head}

    if IS_VIT and CFG.layer_decay < 1.0:
        blocks = getattr(model.backbone, 'blocks', [])
        nb_ = len(blocks)
        buckets = {}
        for name, p in model.backbone.named_parameters():
            m_ = re.search(r'blocks\.(\d+)\.', name)
            depth = (int(m_.group(1)) + 1) if m_ else (0 if any(
                k in name for k in ('patch_embed', 'pos_embed', 'cls_token', 'reg_token')) else nb_)
            buckets.setdefault(round(CFG.layer_decay ** (nb_ - depth), 5), []).append(p)
        groups = [{'params': ps, 'lr': CFG.backbone_lr * s} for s, ps in buckets.items()]
        R.kv(**{'LR бэкбона': f'{CFG.backbone_lr * min(buckets):.2e} … {CFG.backbone_lr:.2e} '
                              f'(затухание {CFG.layer_decay}, блоков {nb_})'})
    else:
        groups = [{'params': [p for p in model.backbone.parameters()], 'lr': CFG.backbone_lr}]
    groups.append({'params': head, 'lr': CFG.lr * CFG.head_lr_mult})
    opt = torch.optim.AdamW(groups, weight_decay=CFG.weight_decay)

    steps = CFG.epochs * max(len(ltr) // CFG.accum, 1)
    warm = int(CFG.warmup_frac * steps)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: s / max(warm, 1) if s < warm
        else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(steps - warm, 1))))

    pos = tr[CFG.target_cols].sum(0).values.astype(np.float32)
    pw = np.clip((len(tr) - pos) / np.clip(pos, 1, None), 1.0, 20.0)
    bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pw, device=CFG.device))
    asl = AsymmetricLoss()
    scaler = GradScaler(enabled=CFG.amp)

    snaps, best, start_ep, hist, gstep = [], -1, 0, [], 0
    ck = load_ckpt(fold)
    if ck is not None:
        model.load_state_dict(_to_fp32(ck['model']))
        if ck.get('ema') is not None:
            ema.load_state_dict(_to_fp32(ck['ema']))
        try:
            opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched'])
            scaler.load_state_dict(ck['scaler'])
        except Exception as e:
            R.warn(f'состояние оптимизатора не восстановлено: {e}')
        snaps = [(s, _to_fp32(sd)) for s, sd in ck['snaps']]
        best, start_ep, hist = ck['best'], ck['epoch'] + 1, ck.get('hist', [])
        set_rng_state(ck['rng'])
        R.ok(f'продолжаем с эпохи {start_ep + 1}/{CFG.epochs} (лучшая пока {best:.3f})')

    yv = va[CFG.target_cols].values
    pred, ep_times = None, []

    for ep in range(start_ep, CFG.epochs):
        t0 = time.time(); model.train()
        tot, nb_i = 0.0, 0
        opt.zero_grad(set_to_none=True)
        pbar = tqdm(ltr, desc=f'{CFG.run_tag} f{fold} ep{ep + 1:>2}/{CFG.epochs}', leave=False)
        for it, b in enumerate(pbar):
            v = b['views'].to(CFG.device, non_blocking=True)
            f = b['feats'].to(CFG.device, non_blocking=True)
            y = b['y'].to(CFG.device, non_blocking=True)
            y = y * (1 - CFG.label_smooth) + 0.5 * CFG.label_smooth
            if CFG.mixup > 0 and random.random() < 0.5:
                lam = float(np.random.beta(CFG.mixup, CFG.mixup))
                pm = torch.randperm(v.size(0), device=v.device)
                v = lam * v + (1 - lam) * v[pm]
                f = lam * f + (1 - lam) * f[pm]
                y = lam * y + (1 - lam) * y[pm]
            with autocast(enabled=CFG.amp):
                logits = model(v, f)
                loss = (1 - CFG.asl_weight) * bce(logits, y) + CFG.asl_weight * asl(logits, y)
            scaler.scale(loss / CFG.accum).backward()
            if (it + 1) % CFG.accum == 0 or it + 1 == len(ltr):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
                sched.step(); gstep += 1; ema.update(model, gstep)
            tot += float(loss.detach()); nb_i += 1
            if it % 20 == 0:
                mem = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0
                pbar.set_postfix_str(f'L={tot / nb_i:.3f} lr={opt.param_groups[-1]["lr"]:.1e} '
                                     f'{mem:.1f}G')

        prob, _ = predict(model, lva, tta=1, desc=f'val ep{ep + 1}')
        pred = (prob > quick_thresholds(yv, prob)).astype(int)
        sc, fq, fa = competition_metric(yv, pred)
        sc_ema = np.nan
        if ep >= 1:
            pe, _ = predict(ema.ema, lva, tta=1, desc=f'val-EMA ep{ep + 1}')
            prd = (pe > quick_thresholds(yv, pe)).astype(int)
            sc_ema, fqe, fae = competition_metric(yv, prd)
            if sc_ema > sc:
                prob, pred, sc, fq, fa = pe, prd, sc_ema, fqe, fae

        dt = time.time() - t0; ep_times.append(dt)
        left = (CFG.epochs - ep - 1) * np.median(ep_times)
        src = 'EMA' if (not np.isnan(sc_ema) and sc_ema == sc) else 'raw'
        print(f'  ep{ep + 1:>2}/{CFG.epochs} | метрика {sc:6.3f} ({src}) = quality {10 * fq:5.2f} '
              f'+ artefacts {10 * fa:5.2f} | loss {tot / nb_i:.3f} | {dt / 60:.1f} мин | '
              f'ост. {_fmt_eta(left)}' + (' <-- best' if sc > best else ''))
        pcf = per_class_f1(yv, pred)
        print('        ' + '  '.join(f'{c[:5]}={pcf[c]:.2f}' for c in CFG.target_cols))

        row = {'run': CFG.run_tag, 'fold': fold, 'epoch': ep + 1, 'train_loss': tot / nb_i,
               'val_metric': sc, 'val_metric_ema': sc_ema, 'f1_quality': fq,
               'f1_artefacts': fa, 'sec': dt, 'time': time.strftime('%Y-%m-%d %H:%M:%S'),
               **{f'f1_{c}': float(v_) for c, v_ in pcf.items()}}
        log_epoch(row); hist.append(row)

        if ep >= 1:
            keep = ema.ema if src == 'EMA' else model
            snaps.append((sc, {k: v_.detach().cpu().clone() for k, v_ in keep.state_dict().items()}))
            snaps.sort(key=lambda x: -x[0]); del snaps[CFG.n_snapshots:]
        best = max(best, sc)

        if (ep + 1) % CFG.ckpt_every == 0 or ep == CFG.epochs - 1:
            conv = _to_fp16 if CFG.ckpt_fp16 else (lambda x: x)
            save_ckpt(fold, {'epoch': ep, 'best': best, 'rng': rng_state(), 'hist': hist,
                             'model': conv(model.state_dict()), 'ema': conv(ema.state_dict()),
                             'opt': opt.state_dict(), 'sched': sched.state_dict(),
                             'scaler': scaler.state_dict(),
                             'snaps': [(s, conv(sd)) for s, sd in snaps],
                             'cfg': {'tile': CFG.tile, 'backbone': CFG.backbone,
                                     'epochs': CFG.epochs, 'feat_ver': FEAT_VER}})

    live_dashboard(hist, fold)
    if not snaps:
        snaps = [(best, {k: v_.detach().cpu().clone() for k, v_ in model.state_dict().items()})]

    lte = DataLoader(MeshViewDataset(test_df.assign(**{c: 0 for c in CFG.target_cols}),
                                     IMG_TEST, feat_te, train=False, labels=False),
                     batch_size=CFG.batch_size * 2, num_workers=CFG.num_workers, pin_memory=True)
    oa, ta = 0.0, 0.0
    for k, (s_, st) in enumerate(snaps):
        model.load_state_dict(st)
        op, oof_ids = predict(model, lva, desc=f'OOF snap{k + 1}/{len(snaps)}')
        tp, te_ids = predict(model, lte, desc=f'test snap{k + 1}/{len(snaps)}')
        oa = oa + op; ta = ta + tp
    oof_prob, te_prob = oa / len(snaps), ta / len(snaps)
    ens = competition_metric(yv, (oof_prob > quick_thresholds(yv, oof_prob)).astype(int))[0]
    print(f'  лучшая эпоха {best:.3f} -> снапшот-ансамбль + TTA {ens:.3f} ({ens - best:+.3f})')
    R.bar(ens, label=f'фолд {fold}')

    save_artifact(rt(f'fold{fold}_preds'),
                  {'oof': oof_prob, 'oof_ids': list(oof_ids), 'te': te_prob,
                   'te_ids': list(te_ids), 'best': best, 'ens': ens})
    drop_ckpt(fold)
    del model, ema, ltr, lva, snaps; gc.collect(); torch.cuda.empty_cache()
    return (oof_ids, oof_prob), (te_ids, te_prob)


In [ ]:
def run_configuration(cfg, label):
    """Полный прогон одной конфигурации. Всё посчитанное берётся с диска:
    завершённый прогон — за секунду, прерванный фолд — с той же эпохи."""
    global IMG_TRAIN, IMG_TEST, feat_tr, feat_te

    apply_config(cfg)
    R.section(f'ПРОГОН {label}', f'{CFG.run_tag} | {CFG.backbone.split(".")[0]} @ {CFG.tile}')
    art = f'{CFG.run_tag}_cnn_preds'
    if CFG.resume and has_artifact(art):
        d = load_artifact(art)
        R.ok(f'{CFG.run_tag} уже завершён (OOF {d["score"]:.3f}) — беру с диска')
        return d

    IMG_TRAIN, _ = build_image_cache(train_df['item_id'].tolist(), TRAIN_DIR, 'train')
    IMG_TEST,  _ = build_image_cache(test_df['item_id'].tolist(),  TEST_DIR,  'test')
    feat_tr, feat_te = assemble_features(IMG_TRAIN, IMG_TEST, CFG.tile)
    if not globals().get('_LAYOUT_OK'):
        check_view_layout(); globals()['_LAYOUT_OK'] = True

    oof = pd.DataFrame({'item_id': train_df['item_id']})
    for c in CFG.target_cols:
        oof['cnn_' + c] = np.nan
    te_parts, fold_scores = [], {}
    for n, fold in enumerate(CFG.folds_to_run, 1):
        (oids, op), (tids, tp) = train_fold(fold)
        oof.loc[oof.set_index('item_id').index.get_indexer(oids),
                ['cnn_' + c for c in CFG.target_cols]] = op
        te_parts.append(pd.DataFrame(tp, columns=['cnn_' + c for c in CFG.target_cols])
                        .assign(item_id=tids))
        yk = train_df.set_index('item_id').loc[oids, CFG.target_cols].values
        fold_scores[fold] = competition_metric(yk, (op > quick_thresholds(yk, op)).astype(int))[0]
        print(f'>>> {CFG.run_tag}: {n}/{len(CFG.folds_to_run)} фолдов | '
              f'от старта {(time.time() - RUN_T0) / 3600:.1f} ч')

    te = pd.concat(te_parts).groupby('item_id', as_index=False).mean()
    m = oof['cnn_quality'].notna().values
    y = train_df.loc[m, CFG.target_cols].values
    p = oof.loc[m, ['cnn_' + c for c in CFG.target_cols]].values
    score = competition_metric(y, (p > quick_thresholds(y, p)).astype(int))[0]
    out = {'oof': oof, 'te': te, 'mask': m, 'score': score, 'folds': fold_scores,
           'tag': CFG.run_tag, 'backbone': CFG.backbone, 'tile': CFG.tile}
    save_artifact(art, out)

    R.kv(**{'OOF метрика': f'{score:.3f}',
            'по фолдам': ', '.join(f'{k}:{v:.2f}' for k, v in fold_scores.items())})
    R.bar(score, label=f'прогон {label}')
    pc = per_class_f1(y, (p > quick_thresholds(y, p)).astype(int))
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    ax[0].barh(pc.index, pc.values,
               color=[PAL['bad'] if v < .4 else PAL['warn'] if v < .6 else PAL['good']
                      for v in pc.values])
    ax[0].set_xlim(0, 1); ax[0].set_title(f'{CFG.run_tag}: per-class F1')
    ax[1].bar([str(k) for k in fold_scores], list(fold_scores.values()), color=PAL['main'])
    ax[1].axhline(np.mean(list(fold_scores.values())), color=PAL['dark'], ls='--')
    ax[1].set_title('по фолдам'); ax[1].set_xlabel('fold')
    plt.tight_layout(); plt.show()
    gc.collect(); torch.cuda.empty_cache()
    return out


def load_reuse_runs():
    """Готовые fold-предсказания прошлых прогонов: ансамблю достаются даром."""
    out, CC = [], ['cnn_' + c for c in CFG.target_cols]
    for tag, pattern in CFG.reuse_runs.items():
        files = [Path(pattern.format(k=k)) for k in range(CFG.n_folds)]
        files = [f for f in files if f.exists()]
        if not files:
            R.warn(f'{tag}: не найдено по шаблону {pattern}'); continue
        o = pd.DataFrame(np.nan, index=train_df['item_id'], columns=CFG.target_cols)
        t = 0
        for f in files:
            d = joblib.load(f)
            o.loc[d['oof_ids'], CFG.target_cols] = d['oof']
            t = t + pd.DataFrame(d['te'], columns=CFG.target_cols, index=d['te_ids']) / len(files)
        m = o['quality'].notna().values
        oof = pd.DataFrame({'item_id': train_df['item_id']})
        oof[CC] = o.values
        te = t.reindex(test_df['item_id']).reset_index()
        te.columns = ['item_id'] + CC
        y = train_df.loc[m, CFG.target_cols].values
        p = o.loc[m, CFG.target_cols].values
        sc = competition_metric(y, (p > quick_thresholds(y, p)).astype(int))[0]
        R.ok(f'{tag}: подключён, {len(files)} фолдов, OOF {sc:.3f}')
        out.append({'oof': oof, 'te': te, 'mask': m, 'score': sc, 'tag': tag,
                    'backbone': 'reuse', 'tile': '-', 'folds': {}})
    return out


def fold_health(run):
    """Расходившийся фолд даёт вырожденные вероятности. Ловим до того, как он
    испортит ансамбль."""
    CC = ['cnn_' + c for c in CFG.target_cols]
    rows = []
    for k in range(CFG.n_folds):
        m = (train_df['fold'] == k).values & run['mask']
        if m.sum() == 0:
            rows.append({'fold': k, 'статус': 'нет данных'}); continue
        P = run['oof'].loc[m, CC].values
        Y = train_df.loc[m, CFG.target_cols].values
        sc = competition_metric(Y, (P > quick_thresholds(Y, P)).astype(int))[0]
        bad = (not np.isfinite(P).all()) or P.std(0).mean() < 0.01 or sc < 5
        rows.append({'fold': k, 'метрика': round(sc, 3),
                     'std': round(float(P.std(0).mean()), 4),
                     'статус': 'РАСХОДИЛСЯ' if bad else 'ок'})
    return pd.DataFrame(rows)


## Инструменты ансамбля

In [ ]:
# ============ Инструменты ансамбля: сборка, правило, согласование ============
import itertools

RESULTS = Path(CFG.drive_dir) / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)

MIN_COVER = 6000          # недоученный прогон сжимает пересечение масок до нуля
KNOWN_TAGS = ['vitb336', 'eva336', 'cnxb336', 'swinb384', 'vitb448',
              'dino448_mine', 'dino448_s2']


def _fit_rule_q(prob, Y, q_src):
    """Пороги по дефектам + стратегия quality. Для взвешенного F1 пороги классов
    независимы, поэтому оптимизируются раздельно."""
    th = np.array([exact_f1_threshold(Y[:, i], prob[:, i])[0] for i in range(10)])
    pd_ = (prob[:, :10] > th).astype(int)
    fa = f1_score(Y[:, :10], pd_, average='weighted', zero_division=0)
    soft = np.prod(1 - np.clip(prob[:, :10], 0, 1), 1)
    hard = (pd_.sum(1) == 0).astype(float)
    best = (-1, .5, .5, 'mix')
    for a in np.arange(0, 1.01, .05):
        pq = a * q_src + (1 - a) * soft
        t, v = exact_f1_threshold(Y[:, 10], pq)
        if v > best[0]:
            best = (v, float(a), float(t), 'mix')
    for mode in ('and', 'or'):
        for a in np.arange(0, 1.01, .1):
            pq = a * q_src + (1 - a) * soft
            t, _ = exact_f1_threshold(Y[:, 10], pq)
            q = ((pq > t) * hard if mode == 'and' else np.maximum((pq > t), hard))
            v = f1_score(Y[:, 10], q.astype(int), zero_division=0)
            if v > best[0]:
                best = (v, float(a), float(t), mode)
    return {'th': th, 'q_alpha': best[1], 'q_th': best[2], 'q_mode': best[3]}


def _apply_rule_q(prob, q_src, RL):
    pd_ = (prob[:, :10] > RL['th']).astype(int)
    soft = np.prod(1 - np.clip(prob[:, :10], 0, 1), 1)
    hard = (pd_.sum(1) == 0).astype(int)
    pq = RL['q_alpha'] * q_src + (1 - RL['q_alpha']) * soft
    qh = (pq > RL['q_th']).astype(int)
    q = {'mix': qh, 'and': qh * hard, 'or': np.maximum(qh, hard)}[RL['q_mode']]
    return np.concatenate([pd_, q[:, None]], 1)


def _consistency(pred, prob, th):
    """quality = 1 ровно тогда, когда нет ни одного дефекта. Объекту, названному
    плохим без единого дефекта, дописываем самый вероятный по отношению к порогу.
    Единственный из шести проверенных вариантов, давший плюс на лидерборде
    (14.987 -> 15.0016). both, top2, взвешивание по классу и q_from_defects — хуже."""
    p = pred.copy()
    rows = np.where((p[:, 10] == 0) & (p[:, :10].sum(1) == 0))[0]
    if len(rows):
        r = np.nan_to_num(prob[rows, :10] / np.maximum(np.asarray(th)[:10], 1e-6), nan=-1.0)
        p[rows, np.argmax(r, 1)] = 1
    return p, len(rows)


def _lmix(arrs, w):
    """Усреднение логитов: среднее вероятностей стягивает уверенные предсказания
    к середине, среднее логитов сохраняет уверенность при согласии моделей."""
    w = np.asarray(w, float); w = w / w.sum()
    z = sum(wi * np.log(np.clip(a, 1e-6, 1 - 1e-6) / (1 - np.clip(a, 1e-6, 1 - 1e-6)))
            for wi, a in zip(w, arrs))
    return 1 / (1 + np.exp(-z))


def _rebuild_healthy(tag):
    """Собирает прогон только из здоровых фолдов и пересобирает усреднение по тесту.
    У схлопнувшегося фолда предсказания на тесте тянут вероятности к нулю, а маска
    действует только на OOF — поэтому усреднение приходится пересобирать."""
    files = {k: ART / f'{tag}_fold{k}_preds.joblib' for k in range(CFG.n_folds)}
    files = {k: p for k, p in files.items() if p.exists()}
    if not files:
        return None
    CC = ['cnn_' + c for c in CFG.target_cols]
    oof = pd.DataFrame({'item_id': train_df['item_id']})
    for c in CFG.target_cols:
        oof['cnn_' + c] = np.nan
    mask = np.zeros(len(train_df), bool)
    good, te_acc, te_ids = [], 0, None
    for k, p in files.items():
        d = joblib.load(p)
        P = np.nan_to_num(np.asarray(d['oof']), nan=0.0, posinf=1.0, neginf=0.0)
        if np.nanstd(P, 0).mean() < 0.05:
            continue
        good.append(k)
        idx = oof.set_index('item_id').index.get_indexer(d['oof_ids'])
        oof.loc[idx, CC] = P
        mask[idx] = True
        te_acc = te_acc + np.nan_to_num(np.asarray(d['te']), nan=0.0, posinf=1.0, neginf=0.0)
        te_ids = d['te_ids']
    if not good:
        R.fail(f'{tag}: все фолды схлопнулись'); return None
    te = pd.DataFrame(te_acc / len(good), columns=CC).assign(item_id=list(te_ids))
    Y = train_df.loc[mask, CFG.target_cols].values
    P = oof.loc[mask, CC].values
    sc = competition_metric(Y, (P > quick_thresholds(Y, P)).astype(int))[0]
    if len(good) < len(files):
        R.warn(f'{tag}: здоровых фолдов {len(good)}/{len(files)} -> {good}')
    return {'oof': oof, 'te': te, 'mask': mask, 'score': sc, 'tag': tag, 'folds': {}}


def load_reuse_runs():
    """Готовые fold-предсказания прошлых прогонов: ансамблю достаются даром."""
    out, CC = [], ['cnn_' + c for c in CFG.target_cols]
    for tag, pattern in CFG.reuse_runs.items():
        files = [Path(pattern.format(k=k)) for k in range(CFG.n_folds)]
        files = [f for f in files if f.exists()]
        if not files:
            R.warn(f'{tag}: не найдено по шаблону {pattern}'); continue
        o = pd.DataFrame(np.nan, index=train_df['item_id'], columns=CFG.target_cols)
        t = 0
        for f in files:
            d = joblib.load(f)
            o.loc[d['oof_ids'], CFG.target_cols] = np.nan_to_num(np.asarray(d['oof']), nan=0.0)
            t = t + pd.DataFrame(np.nan_to_num(np.asarray(d['te']), nan=0.0),
                                 columns=CFG.target_cols, index=d['te_ids']) / len(files)
        m = o['quality'].notna().values
        oof = pd.DataFrame({'item_id': train_df['item_id']}); oof[CC] = o.values
        te = t.reindex(test_df['item_id']).reset_index(); te.columns = ['item_id'] + CC
        Y = train_df.loc[m, CFG.target_cols].values
        P = o.loc[m, CFG.target_cols].values
        sc = competition_metric(Y, (P > quick_thresholds(Y, P)).astype(int))[0]
        R.ok(f'{tag}: подключён, {len(files)} фолдов, OOF {sc:.3f}')
        out.append({'oof': oof, 'te': te, 'mask': m, 'score': sc, 'tag': tag, 'folds': {}})
    return out


def collect_runs():
    runs, seen = [], set()
    for tag in KNOWN_TAGS:
        r = _rebuild_healthy(tag)
        if r is not None and r['tag'] not in seen:
            runs.append(r); seen.add(r['tag'])
    for r in load_reuse_runs():
        if r['tag'] not in seen:
            runs.append(r); seen.add(r['tag'])
    ok = []
    for r in runs:
        cov = int(r['mask'].sum())
        if r['score'] < 13.0:
            R.warn(f'{r["tag"]}: OOF {r["score"]:.3f} < 13.0 — исключён')
        elif cov < MIN_COVER:
            R.warn(f'{r["tag"]}: покрытие {cov} (ещё учится) — пока исключён')
        else:
            ok.append(r)
    return ok


def build_and_save(label):
    """Ансамбль всего доступного -> веса по OOF -> правило -> согласование -> сабмит.
    Вызывается после каждой модели, чтобы готовый файл существовал в любой момент."""
    runs = collect_runs()
    if not runs:
        R.fail('нет прогонов'); return None
    CC = ['cnn_' + c for c in CFG.target_cols]
    mask = np.logical_and.reduce([r['mask'] for r in runs])
    if mask.sum() < 1000:
        R.fail(f'пересечение масок {int(mask.sum())} — поднимите MIN_COVER'); return None
    Y = train_df.loc[mask, CFG.target_cols].values
    folds = train_df.loc[mask, 'fold'].values
    O = [np.nan_to_num(r['oof'].loc[mask, CC].values, nan=0.0) for r in runs]
    T = [np.nan_to_num(r['te'].set_index('item_id').reindex(test_df['item_id'])[CC].values,
                       nan=0.0) for r in runs]
    names = [r['tag'] for r in runs]

    def sc(P):
        return competition_metric(Y, (P > quick_thresholds(Y, P)).astype(int))[0]

    solo = [sc(o) for o in O]
    step = (0, .25, .5, .75, 1.) if len(O) <= 3 else (0, .5, 1.)
    bw, bs = None, -1
    for w in itertools.product(step, repeat=len(O)):
        if sum(w) == 0:
            continue
        s = sc(_lmix(O, w))
        if s > bs:
            bs, bw = s, w
    # Равные веса однажды смешали модель на 13.8 с моделью на 12.1 поровну и увели
    # результат ниже лучшей одиночной. Поэтому веса подбираются.
    if bs <= max(solo) + 0.02:
        bi = int(np.argmax(solo)); bw = tuple(int(i == bi) for i in range(len(O)))
    p_o, p_t = _lmix(O, bw), _lmix(T, bw)
    q_o = _lmix([o[:, [10]] for o in O], bw)[:, 0]
    q_t = _lmix([t[:, [10]] for t in T], bw)[:, 0]

    RL = _fit_rule_q(p_o, Y, q_o)
    pred_o, _ = _consistency(_apply_rule_q(p_o, q_o, RL), p_o, RL['th'])
    S, fq, fa = competition_metric(Y, pred_o)
    nst = []
    for k in np.unique(folds):
        tr_, va_ = folds != k, folds == k
        Rk = _fit_rule_q(p_o[tr_], Y[tr_], q_o[tr_])
        pk, _ = _consistency(_apply_rule_q(p_o[va_], q_o[va_], Rk), p_o[va_], Rk['th'])
        nst.append(competition_metric(Y[va_], pk)[0])
    nst = np.array(nst)

    pred_t, n_fix = _consistency(_apply_rule_q(p_t, q_t, RL), p_t, RL['th'])
    sub = pd.DataFrame(pred_t, columns=CFG.target_cols)
    sub.insert(0, 'item_id', test_df['item_id'].values)
    sub = sub[['item_id'] + CFG.target_cols]
    assert len(sub) == len(test_df) and sub['item_id'].is_unique
    stamp = time.strftime('%H%M')
    sub.to_csv(RESULTS / f'submission_{label}_{stamp}.csv', index=False)
    sub.to_csv(RESULTS / 'submission_best.csv', index=False)
    sub.to_csv('submission_best.csv', index=False)
    with open(RESULTS / f'summary_{label}_{stamp}.json', 'w', encoding='utf-8') as fh:
        json.dump({'label': label, 'время': time.strftime('%Y-%m-%d %H:%M'),
                   'вложенная': round(float(nst.mean()), 4),
                   'вложенная_std': round(float(nst.std()), 4),
                   'oof': round(float(S), 4),
                   'quality': round(10 * float(fq), 3), 'artefacts': round(10 * float(fa), 3),
                   'прогоны': names,
                   'одиночные': {n: round(float(x), 3) for n, x in zip(names, solo)},
                   'веса': {n: float(w) for n, w in zip(names, bw)},
                   'согласовано объектов': int(n_fix),
                   'меток': int(pred_t[:, :10].sum())}, fh, ensure_ascii=False, indent=2)

    R.section(f'СБОРКА: {label}', ', '.join(names))
    R.kv(**{'одиночные': dict(zip(names, [round(x, 3) for x in solo])),
            'веса': dict(zip(names, bw)), 'покрытие': int(mask.sum()),
            'OOF': f'{S:.3f}', 'ВЛОЖЕННАЯ': f'{nst.mean():.3f} ± {nst.std():.3f}',
            'согласование': f'дописан дефект {n_fix} объектам'})
    R.bar(nst.mean(), 14, 16, label='вложенная')
    if len(O) >= 2:
        E = [np.abs(o - Y) for o in O]
        rr = pd.DataFrame([[np.mean([np.corrcoef(E[i][:, c], E[j][:, c])[0, 1]
                                     for c in range(11)]) for j in range(len(E))]
                           for i in range(len(E))], index=names, columns=names)
        print('корреляция ошибок (ниже 0.7 — источники дополняют друг друга):')
        display(rr.round(2))
    return {'nested': float(nst.mean()), 'names': names}


## §9. Три источника ансамбля

Каждый источник — отдельная конфигурация «бэкбон + разрешение». В режиме
`USE_CACHE = True` готовые прогоны подключаются с Drive; при `USE_CACHE = False`
все три обучаются кодом ниже с нуля.

Обучение возобновляемое: завершённый фолд берётся с диска, прерванный продолжается
с той же эпохи с восстановлением состояния генератора случайных чисел. В каждой эпохе
печатается метрика, её разложение на `quality` и `artefacts`, per-class F1 и остаток
времени; прогресс-бар показывает loss, learning rate и занятую память GPU.

### 9.1. ViT-B/14 @ 336

In [ ]:
# ============ §9.1. Источник 1: DINOv2 ViT-B/14 @ 336 ============
# 86M параметров против 22M у Small, разрешение 336 против 224 у базового прогона:
# 576 токенов на вид вместо 256. Обучается кодом этого ноутбука в обоих режимах.
if not USE_CACHE:
    CFG.force_recompute += [f'vitb336_fold{k}_preds' for k in range(CFG.n_folds)]
    CFG.force_recompute += ['vitb336_cnn_preds']

RUN_VITB = run_configuration(CONFIG_VITB, 'ViT-B/14 @ 336')


### 9.2. ViT-B/14 @ 448

In [ ]:
# ============ §9.2. Источник 2: DINOv2 ViT-B/14 @ 448 ============
# Сильнейший одиночный источник ансамбля (OOF 14.685, вес 1.0). Тот же бэкбон,
# что и §9.1, но 1024 токена на вид: разрешение ловит текстурные дефекты
# (noisy, lowpoly), которые на 336 размываются.
import zipfile

if USE_CACHE:
    # Прогон обучался в отдельной сессии, предсказания фолдов лежат в архиве на Drive.
    # Из архива берём ТОЛЬКО *_preds.joblib: веса (2 ГБ) для сборки не нужны.
    EXT = CFG.work / 'imported'; EXT.mkdir(parents=True, exist_ok=True)
    n_new = 0
    for z in sorted(Path('/content/drive/MyDrive').rglob(IMPORT_ZIP_GLOB)):
        with zipfile.ZipFile(z) as f:
            for m in [m for m in f.namelist()
                      if m.endswith('.joblib') and 'fold' in m and 'preds' in m]:
                dst = ART / Path(m).name
                if dst.exists():
                    continue
                f.extract(m, EXT); shutil.copy2(EXT / m, dst); push_to_drive(dst)
                n_new += 1
    n_have = len(list(ART.glob('dino448_s2_fold*_preds.joblib')))
    R.kv(**{'подключено новых файлов': n_new, 'фолдов доступно': f'{n_have}/{CFG.n_folds}'})
    assert n_have > 0, f'не найден архив {IMPORT_ZIP_GLOB} на Drive'

    # Разбиение на фолды обязано совпадать, иначе OOF-предсказания несравнимы
    # по объектам и подбор весов пойдёт по мусору.
    d = joblib.load(ART / 'dino448_s2_fold0_preds.joblib')
    mine = set(train_df.loc[train_df.fold == 0, 'item_id'])
    ov = len(set(d['oof_ids']) & mine) / max(len(d['oof_ids']), 1)
    print(f'совпадение фолда 0 с нашим разбиением: {100 * ov:.0f}%')
    assert ov > 0.95, 'разбиение на фолды отличается — импорт использовать нельзя'
    RUN_448 = _rebuild_healthy('dino448_s2')
    R.ok(f'dino448_s2: OOF {RUN_448["score"]:.3f}, покрытие {int(RUN_448["mask"].sum())}')
else:
    CFG.force_recompute += [f'dino448_s2_fold{k}_preds' for k in range(CFG.n_folds)]
    CFG.force_recompute += ['dino448_s2_cnn_preds']
    RUN_448 = run_configuration(CONFIG_448, 'ViT-B/14 @ 448')


### 9.3. ViT-S/14 @ 224

In [ ]:
# ============ §9.3. Источник 3: DINOv2 ViT-S/14 @ 224 ============
# Базовый прогон: вчетверо меньше параметров и вдвое ниже разрешение. В одиночку
# слабейший из трёх (OOF 14.376), но ошибается иначе — корреляция с остальными
# 0.77 и 0.84, самая низкая пара в ансамбле. Вес 0.5.
if USE_CACHE:
    src = sorted(Path('/content/drive/MyDrive').glob(IMPORT_V5_GLOB.format(k='*')))
    print(f'найдено файлов: {len(src)}')
    assert src, f'не найдены предсказания по шаблону {IMPORT_V5_GLOB}'
    for i, p in enumerate(src):
        dst = ART / f'v5_224_fold{i}_preds.joblib'
        if not dst.exists():
            shutil.copy2(p, dst); push_to_drive(dst)
            print(f'  подключён {p.name} -> {dst.name}')
    d = joblib.load(ART / 'v5_224_fold0_preds.joblib')
    mine = set(train_df.loc[train_df.fold == 0, 'item_id'])
    ov = len(set(d['oof_ids']) & mine) / max(len(d['oof_ids']), 1)
    print(f'совпадение фолда 0 с нашим разбиением: {100 * ov:.0f}%')
    assert ov > 0.95, 'разбиение на фолды отличается'
    RUN_V5 = _rebuild_healthy('v5_224')
    R.ok(f'v5_224: OOF {RUN_V5["score"]:.3f}, покрытие {int(RUN_V5["mask"].sum())}')
else:
    CFG.force_recompute += [f'v5_224_fold{k}_preds' for k in range(CFG.n_folds)]
    CFG.force_recompute += ['v5_224_cnn_preds']
    RUN_V5 = run_configuration(CONFIG_V5, 'ViT-S/14 @ 224')


## §10. Ансамбль

Смешивание в logit-пространстве с весами, подобранными перебором по OOF. Равное
усреднение не используется: в одном из прогонов оно смешало модель на 13.8 с моделью
на 12.1 поровну и увело результат **ниже** лучшей одиночной.

Матрица корреляции ошибок показывает, дополняют ли источники друг друга.

In [ ]:
# ============ §10. Ансамбль трёх источников ============
R.section('Ансамбль', 'веса подбираются по OOF, лидерборд не участвует')

KNOWN_TAGS = FINAL_TAGS
MIN_COVER = 4000          # ниже этого пороги настраиваются на слишком шумной выборке


def collect_runs():
    """Ровно три источника финального состава — ничего лишнего в бленд не попадёт."""
    out = []
    for tag in FINAL_TAGS:
        r = _rebuild_healthy(tag)
        if r is None:
            R.fail(f'{tag}: предсказаний нет'); continue
        if int(r['mask'].sum()) < MIN_COVER:
            R.warn(f'{tag}: покрытие {int(r["mask"].sum())} < {MIN_COVER}')
        out.append(r)
    return out


runs = collect_runs()
assert len(runs) == 3, f'нужны все три источника, есть {len(runs)}'
for r in runs:
    print(f'  {r["tag"]:<14s} OOF {r["score"]:.3f} | покрытие {int(r["mask"].sum())}')
cov = int(np.logical_and.reduce([r['mask'] for r in runs]).sum())
print(f'пересечение: {cov} объектов')

RES = build_and_save('final_15541')
assert RES is not None
R.kv(**{'ВЛОЖЕННАЯ ОЦЕНКА': f'{RES["nested"]:.3f}',
        'ожидание на лидерборде': f'{RES["nested"]:.2f} ± 0.4 (669 объектов теста)'})


## §12. submission.csv и проверки формата

In [ ]:
# ============ Финальная сборка и submission.csv ============
R.section('submission.csv')

sub = pd.read_csv(RESULTS / 'submission_best.csv')
sub.to_csv('submission.csv', index=False)

ok = True
for name, res in [
        ('строк ровно как объектов теста', len(sub) == len(test_df)),
        ('item_id уникальны', sub['item_id'].is_unique),
        ('набор колонок совпадает с train', set(sub.columns) == set(pd.read_csv(TRAIN_CSV).columns)),
        ('только 0 и 1', set(np.unique(sub[CFG.target_cols].values)) <= {0, 1}),
        ('нет пропусков', not sub.isna().any().any())]:
    print(f'  {"[OK] " if res else "[FAIL]"} {name}'); ok &= bool(res)
assert ok, 'submission.csv не прошёл проверку'

X = sub[CFG.target_cols].values
nd = X[:, :10].sum(1)
R.kv(**{'submission.csv': f'{len(sub)} строк', 'меток': int(nd.sum()),
        'quality=1': f"{int(X[:, 10].sum())} ({X[:, 10].mean():.3f}, "
                     f"в train {train_df['quality'].mean():.3f})",
        'противоречий': int(((X[:, 10] == 0) & (nd == 0)).sum()
                            + ((X[:, 10] == 1) & (nd > 0)).sum()),
        'ВЛОЖЕННАЯ ОЦЕНКА': f'{RES["nested"]:.3f}'})
R.bar(RES['nested'], 14, 16, label='вложенная')

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
x = np.arange(11)
ax[0].bar(x - .2, train_df[CFG.target_cols].mean(), .4, label='train', color=PAL['dark'])
ax[0].bar(x + .2, X.mean(0), .4, label='сабмит', color=PAL['good'])
ax[0].set_xticks(x); ax[0].set_xticklabels(CFG.target_cols, rotation=80, fontsize=8)
ax[0].legend(fontsize=8); ax[0].set_title('Доли меток')
cnt = pd.Series(nd).value_counts().sort_index()
ax[1].bar(cnt.index, cnt.values, color=PAL['main'])
ax[1].set_xlabel('дефектов у объекта'); ax[1].set_title('Распределение числа дефектов')
plt.tight_layout(); plt.show()
display(sub.head())


## §13. Визуализация решений

Три независимых метода, каждый отвечает на свой вопрос.

1. **Карта внимания ML-Decoder** — на какие участки каких ракурсов смотрел запрос
   конкретного класса. Обычный Grad-CAM даёт одну карту на всё предсказание,
   а здесь карта своя у каждой метки.
2. **Разложение ошибки по классам** — где именно теряются баллы, с учётом веса класса
   в метрике. Без этого легко улучшать класс ценой 0.01 балла.
3. **Отбор показательных примеров** — уверенные попадания и уверенные промахи.
   Смотреть надо именно на уверенные ошибки: неуверенные объясняются порогом,
   а уверенные указывают на дефект модели или разметки.

In [ ]:
# ============ Визуализация решений: три метода, fail- и success-кейсы ============
# Три независимых способа показать, НА ЧТО смотрит модель, и где она ошибается.
# Каждый отвечает на свой вопрос, поэтому они дополняют друг друга, а не дублируют.
R.section('Визуализация решений')

# Модель для карт внимания: берём веса первого фолда первой конфигурации.
# Если весов нет (старый прогон удалял чекпоинты), методы 2 и 3 всё равно работают —
# они считаются по сохранённым предсказаниям.
_VIZ_MODEL = None
_w = sorted(CKPT.glob(f'{CONFIGS[0]["run_tag"]}_cnn_fold*.pth'))
if _w:
    apply_config(CONFIGS[0])
    _VIZ_MODEL = MultiViewNet(len(FEAT_COLS)).to(CFG.device).eval()
    _VIZ_MODEL.load_state_dict(_to_fp32(torch.load(_w[0], map_location='cpu',
                                                   weights_only=False)))
    R.ok(f'модель для карт внимания загружена: {_w[0].name}')
else:
    R.warn('весов нет — метод 1 (карты внимания) пропускается, '
           'методы 2 и 3 работают по предсказаниям')

CC = ['cnn_' + c for c in CFG.target_cols]
VIZ = _rebuild_healthy(CONFIGS[0]['run_tag'])
assert VIZ is not None, 'нет предсказаний для визуализации'
Yv = train_df.loc[VIZ['mask'], CFG.target_cols].values
Pv = VIZ['oof'].loc[VIZ['mask'], CC].values
IDv = train_df.loc[VIZ['mask'], 'item_id'].values
THv = quick_thresholds(Yv, Pv)
PREDv = (Pv > THv).astype(int)


# ---------- метод 1: карта внимания ML-Decoder ----------
def attention_map(item_id, cls, model=None):
    """У ML-Decoder свой обучаемый запрос на каждый класс, и его веса внимания
    показывают, какие участки каких ракурсов повлияли ИМЕННО на этот класс.
    Обычный Grad-CAM так не умеет: он даёт одну карту на всё предсказание."""
    p = Path(IMG_TRAIN) / f'{item_id}.png'
    v = load_views(p, CFG.tile)
    x = ((v / 255.0 - MEAN) / STD).astype(np.float32)
    x = torch.from_numpy(x).permute(0, 3, 1, 2).unsqueeze(0).to(CFG.device)
    f = torch.zeros(1, len(FEAT_COLS), device=CFG.device)
    model = model or _VIZ_MODEL
    with torch.no_grad(), torch.cuda.amp.autocast(enabled=CFG.amp):
        logits, att = model(x, f, return_att=True)
    i = CFG.target_cols.index(cls)
    n = CFG.mld_tokens
    a = att[0, i, :6 * n * n].float().cpu().numpy().reshape(6, n, n)
    return v, a, float(torch.sigmoid(logits[0, i]))


def show_attention(item_id, cls, title=''):
    v, a, p = attention_map(item_id, cls)
    a = (a - a.min()) / (a.max() - a.min() + 1e-9)
    fig, ax = plt.subplots(2, 6, figsize=(16, 5.6))
    for k in range(6):
        ax[0, k].imshow(v[k]); ax[0, k].axis('off')
        ax[1, k].imshow(v[k]); ax[1, k].axis('off')
        ax[1, k].imshow(np.kron(a[k], np.ones((CFG.tile // CFG.mld_tokens,) * 2)),
                        cmap='jet', alpha=0.45, extent=(0, CFG.tile, CFG.tile, 0))
        ax[0, k].set_title(f'вид {k}', fontsize=8)
    ax[0, 0].set_ylabel('кадр', fontsize=9)
    fig.suptitle(f'{title}  |  запрос «{cls}», p = {p:.3f}', fontsize=11, weight='bold')
    plt.tight_layout(); plt.show()


# ---------- метод 2: разложение ошибки по классам ----------
def error_profile():
    """Где именно теряются баллы: вклад каждого класса в метрику против его потолка.
    Без этого легко улучшать класс, который стоит 0.01 балла."""
    w = Yv[:, :10].sum(0) / Yv[:, :10].sum()
    f1 = per_class_f1(Yv, PREDv)
    got = np.append(10 * w * f1.values[:10], 10 * f1['quality'])
    cap = np.append(10 * w, 10.0)
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
    o = np.argsort(cap - got)
    y = np.arange(11)
    ax[0].barh(y, cap[o], color='#e3e6ea', label='потолок')
    ax[0].barh(y, got[o], color=PAL['main'], label='получено')
    ax[0].set_yticks(y); ax[0].set_yticklabels([CFG.target_cols[i] for i in o], fontsize=8)
    ax[0].legend(fontsize=8); ax[0].set_title('Баллы: получено против потолка')

    fp = (PREDv == 1) & (Yv == 0)
    fn = (PREDv == 0) & (Yv == 1)
    x = np.arange(11)
    ax[1].bar(x - .2, fp.sum(0), .4, label='ложные срабатывания', color=PAL['bad'])
    ax[1].bar(x + .2, fn.sum(0), .4, label='пропуски', color=PAL['warn'])
    ax[1].set_xticks(x); ax[1].set_xticklabels(CFG.target_cols, rotation=80, fontsize=8)
    ax[1].legend(fontsize=8); ax[1].set_title('Баланс ошибок по классам')

    q_fp = (PREDv[:, 10] == 1) & (Yv[:, 10] == 0)
    cause = pd.Series(Yv[q_fp][:, :10].mean(0), index=CFG.artifact_cols).sort_values()
    ax[2].barh(cause.index, cause.values, color=PAL['alt'])
    ax[2].set_title(f'Что срывает quality ({int(q_fp.sum())} ошибок)')
    plt.tight_layout(); plt.show()
    print('>>> Ошибки quality производны от детекции дефектов: чаще всего это пропущенные',
          ', '.join(cause.tail(3).index[::-1]))


# ---------- метод 3: отбор показательных примеров ----------
def pick_cases(cls, n=3):
    """Уверенные попадания и уверенные промахи. Смотреть надо именно на уверенные
    ошибки: неуверенные объясняются порогом, а уверенные — дефектом модели."""
    i = CFG.target_cols.index(cls)
    ok = np.where((PREDv[:, i] == 1) & (Yv[:, i] == 1))[0]
    fp = np.where((PREDv[:, i] == 1) & (Yv[:, i] == 0))[0]
    fn = np.where((PREDv[:, i] == 0) & (Yv[:, i] == 1))[0]
    best = ok[np.argsort(-Pv[ok, i])][:n] if len(ok) else []
    worst_fp = fp[np.argsort(-Pv[fp, i])][:n] if len(fp) else []
    worst_fn = fn[np.argsort(Pv[fn, i])][:n] if len(fn) else []
    return best, worst_fp, worst_fn


def show_grid(idx, cls, kind):
    if not len(idx):
        print(f'  {cls}/{kind}: примеров нет'); return
    fig, ax = plt.subplots(len(idx), 6, figsize=(15, 2.5 * len(idx)))
    ax = np.atleast_2d(ax)
    for r, j in enumerate(idx):
        v = load_views(Path(IMG_TRAIN) / f'{IDv[j]}.png', CFG.tile)
        true = [c for c in CFG.artifact_cols if Yv[j, CFG.target_cols.index(c)] == 1]
        for k in range(6):
            ax[r, k].imshow(v[k]); ax[r, k].axis('off')
        ax[r, 0].set_title(f'p={Pv[j, CFG.target_cols.index(cls)]:.2f} | истина: '
                           f'{true or ["нет дефектов"]}', loc='left', fontsize=8)
    fig.suptitle(f'{cls} — {kind}', fontsize=12, weight='bold')
    plt.tight_layout(); plt.show()


error_profile()

if _VIZ_MODEL is not None:
    # По одному примеру на каждый край: где модель уверенно права и уверенно ошибается.
    for cls in ('noisy', 'open'):
        b, fp_, fn_ = pick_cases(cls, n=1)
        if len(b):
            show_attention(IDv[b[0]], cls, 'верное срабатывание')
        if len(fp_):
            show_attention(IDv[fp_[0]], cls, 'ЛОЖНОЕ срабатывание')

# Показываем два класса-антипода: где модель работает и где почти не работает.
for cls in ('noisy', 'open'):
    b, fp_, fn_ = pick_cases(cls)
    print(f'\n{"=" * 70}\nКЛАСС {cls}: F1 = {per_class_f1(Yv, PREDv)[cls]:.3f}')
    show_grid(b, cls, 'уверенные попадания')
    show_grid(fp_, cls, 'уверенные ЛОЖНЫЕ срабатывания (fail)')
    show_grid(fn_, cls, 'уверенные ПРОПУСКИ (fail)')


## §14. Чек-лист воспроизводимости

In [ ]:
# ============ Чек-лист воспроизводимости ============
R.section('Воспроизводимость')

import platform, sklearn
info = {
    'режим': 'кэш' if USE_CACHE else 'полное обучение с нуля',
    'random_seed': CFG.seed,
    'разбиение на фолды': f'MultilabelStratifiedKFold, {CFG.n_folds} фолдов, '
                          f'seed={CFG.seed}, shuffle=True',
    'источники': [c['run_tag'] for c in CONFIGS],
    'бэкбоны': [c['backbone'] for c in CONFIGS],
    'разрешения': [c['tile'] for c in CONFIGS],
    'эпох на фолд': CFG.epochs,
    'loss': f'BCE(pos_weight) {1 - CFG.asl_weight:.2f} + AsymmetricLoss {CFG.asl_weight:.2f}',
    'голова': 'ML-Decoder' if CFG.use_mldecoder else 'gated attention pooling',
    'аугментация': f'{len(SYM_ALL)} симметрий стенда, mixup {CFG.mixup}',
    'TTA': f'{len(CFG.tta_syms)} преобразований',
    'признаков': len(FEAT_COLS),
    'python': platform.python_version(),
    'torch': torch.__version__,
    'timm': __import__('timm').__version__,
    'sklearn': sklearn.__version__,
    'GPU': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
    'время прогона': _fmt_eta(time.time() - RUN_T0),
}
for k, v in info.items():
    print(f'  {k:<26s} {v}')

# Ни один шаг не использует обратную связь лидерборда: веса ансамбля, пороги
# и стратегия quality выводятся исключительно из OOF-предсказаний.
print('\n  [OK] лидерборд не участвует в настройке: всё выводится из OOF')
print('  [OK] seed зафиксирован во всех источниках случайности '
      '(python, numpy, torch, cuda)')
print('  [OK] данные скачиваются из ноутбука (§2), библиотеки ставятся в §0')
print('  [OK] submission.csv прошёл проверки формата (§12)')

with open('reproducibility.json', 'w', encoding='utf-8') as f:
    json.dump({k: (v if isinstance(v, (str, int, float, list)) else str(v))
               for k, v in info.items()} | {'вложенная_оценка': round(RES['nested'], 4)},
              f, ensure_ascii=False, indent=2)
print('\nreproducibility.json сохранён')
